In [2]:
# ============================================================
# STEP 1 — ENVIRONMENT + HAM10000 DATASET AUDIT
# ============================================================

import os
import sys
import glob
import random
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchvision

print("=" * 80)
print("BIASPRUNER — STEP 1")
print("ENVIRONMENT + HAM10000 DATASET AUDIT")
print("=" * 80)


# ============================================================
# 1. PYTHON / PYTORCH / CUDA
# ============================================================

print("\n" + "-" * 80)
print("1. ENVIRONMENT")
print("-" * 80)

print("Python version :", sys.version)
print("PyTorch version:", torch.__version__)
print("Torchvision    :", torchvision.__version__)

cuda_available = torch.cuda.is_available()

print("CUDA available :", cuda_available)

if not cuda_available:
    raise RuntimeError(
        "\nCUDA GPU is NOT available.\n"
        "STOPPING before the expensive BiasPruner experiment."
    )

device = torch.device("cuda")

print("Device         :", device)
print("GPU count      :", torch.cuda.device_count())

for gpu_id in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(gpu_id)

    print(f"\nGPU {gpu_id}")
    print("  Name       :", props.name)
    print("  VRAM       :", round(
        props.total_memory / (1024 ** 3), 2
    ), "GB")
    print("  Compute    :", f"{props.major}.{props.minor}")

print(
    "\nSelected GPU:",
    torch.cuda.get_device_name(0)
)


# ============================================================
# 2. BASIC RUNTIME INFORMATION
# ============================================================

print("\n" + "-" * 80)
print("2. RUNTIME")
print("-" * 80)

print("Working directory:", os.getcwd())
print("Platform         :", platform.platform())


# ============================================================
# 3. SEARCH FOR HAM10000 METADATA
# ============================================================

print("\n" + "-" * 80)
print("3. SEARCHING FOR HAM10000")
print("-" * 80)

# Common Colab locations are searched, but we do NOT assume
# any fixed path.

search_roots = [
    "/content",
    "/mnt/data",
    "/kaggle",
    "/workspace"
]

metadata_candidates = []

for root in search_roots:

    if not os.path.exists(root):
        continue

    for pattern in [
        "**/HAM10000_metadata.csv",
        "**/ham10000_metadata.csv",
        "**/HAM10000_metadata",
        "**/*metadata*.csv"
    ]:

        matches = glob.glob(
            os.path.join(root, pattern),
            recursive=True
        )

        metadata_candidates.extend(matches)


# Remove duplicates
metadata_candidates = sorted(
    set(metadata_candidates)
)

print("\nMetadata candidates:")

for path in metadata_candidates:
    print("  ", path)


if len(metadata_candidates) == 0:

    raise FileNotFoundError(
        "\nHAM10000 metadata CSV was not found.\n"
        "Upload/mount the HAM10000 dataset in Colab and rerun Step 1."
    )


# Prefer exact HAM10000 filename if available
exact_matches = [
    p for p in metadata_candidates
    if Path(p).name.lower() == "ham10000_metadata.csv"
]

if exact_matches:
    metadata_path = exact_matches[0]
else:
    metadata_path = metadata_candidates[0]

print("\nSelected metadata:")
print(metadata_path)


# ============================================================
# 4. LOAD METADATA
# ============================================================

df = pd.read_csv(metadata_path)

print("\nMetadata shape:")
print(df.shape)

print("\nMetadata columns:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())


# ============================================================
# 5. VERIFY REQUIRED COLUMNS
# ============================================================

print("\n" + "-" * 80)
print("4. METADATA SCHEMA CHECK")
print("-" * 80)

required_columns = [
    "image_id",
    "dx",
    "age"
]

missing_columns = [
    c for c in required_columns
    if c not in df.columns
]

if missing_columns:

    raise ValueError(
        f"Missing required metadata columns: {missing_columns}"
    )

print("Required columns: PASS")


# ============================================================
# 6. DATASET BASIC AUDIT
# ============================================================

print("\n" + "-" * 80)
print("5. DATASET AUDIT")
print("-" * 80)

print("Total metadata rows :", len(df))

print(
    "Unique image IDs    :",
    df["image_id"].nunique()
)

print(
    "Duplicate image IDs :",
    int(df["image_id"].duplicated().sum())
)

print("\nClass distribution:")

print(
    df["dx"]
    .value_counts()
    .sort_index()
)


# ============================================================
# 7. MISSING VALUES
# ============================================================

print("\nMissing values:")

print(
    df[
        ["image_id", "dx", "age"]
    ]
    .isna()
    .sum()
)

missing_age = int(df["age"].isna().sum())

missing_label = int(df["dx"].isna().sum())

missing_image_id = int(
    df["image_id"].isna().sum()
)

print("\nMissing age      :", missing_age)
print("Missing labels   :", missing_label)
print("Missing image ID :", missing_image_id)


# ============================================================
# 8. AGE DISTRIBUTION
# ============================================================

print("\n" + "-" * 80)
print("6. AGE DISTRIBUTION")
print("-" * 80)

print(df["age"].describe())

print("\nAge < 60 / Age >= 60:")

age_numeric = pd.to_numeric(
    df["age"],
    errors="coerce"
)

age_groups = pd.Series(
    np.where(
        age_numeric.isna(),
        "missing",
        np.where(
            age_numeric < 60,
            "<60",
            ">=60"
        )
    )
)

print(age_groups.value_counts())


# ============================================================
# 9. SEARCH FOR IMAGE FILES
# ============================================================

print("\n" + "-" * 80)
print("7. SEARCHING FOR IMAGE FILES")
print("-" * 80)

image_extensions = [
    "*.jpg",
    "*.jpeg",
    "*.png",
    "*.JPG",
    "*.JPEG",
    "*.PNG"
]

image_candidates = []

# Search around metadata location first
metadata_parent = str(
    Path(metadata_path).parent
)

search_locations = [
    metadata_parent,
    "/content",
    "/mnt/data"
]

for root in search_locations:

    if not os.path.exists(root):
        continue

    for extension in image_extensions:

        matches = glob.glob(
            os.path.join(
                root,
                "**",
                extension
            ),
            recursive=True
        )

        image_candidates.extend(matches)


image_candidates = sorted(
    set(image_candidates)
)

print(
    "Image files found:",
    len(image_candidates)
)

print("\nExample image paths:")

for path in image_candidates[:10]:
    print("  ", path)


if len(image_candidates) == 0:

    raise FileNotFoundError(
        "\nNo image files were found.\n"
        "HAM10000 images must be available before continuing."
    )


# ============================================================
# 10. IMAGE ↔ METADATA CORRESPONDENCE
# ============================================================

print("\n" + "-" * 80)
print("8. IMAGE / METADATA CORRESPONDENCE")
print("-" * 80)

image_ids_on_disk = {
    Path(path).stem
    for path in image_candidates
}

metadata_ids = set(
    df["image_id"]
    .dropna()
    .astype(str)
)

missing_images = (
    metadata_ids
    - image_ids_on_disk
)

unmatched_images = (
    image_ids_on_disk
    - metadata_ids
)

print(
    "Metadata image IDs:",
    len(metadata_ids)
)

print(
    "Image IDs on disk :",
    len(image_ids_on_disk)
)

print(
    "Metadata IDs with no image:",
    len(missing_images)
)

print(
    "Images with no metadata:",
    len(unmatched_images)
)


# ============================================================
# 11. REQUIRED TASK CLASSES
# ============================================================

print("\n" + "-" * 80)
print("9. REQUIRED TASK CLASSES")
print("-" * 80)

TASK1_CLASSES = ["bkl", "mel"]
TASK2_CLASSES = ["bcc", "akiec"]

required_task_classes = (
    TASK1_CLASSES +
    TASK2_CLASSES
)

print("Task 1:", TASK1_CLASSES)
print("Task 2:", TASK2_CLASSES)

available_classes = set(
    df["dx"]
    .dropna()
    .astype(str)
    .str.lower()
)

missing_task_classes = [
    c for c in required_task_classes
    if c not in available_classes
]

if missing_task_classes:

    raise ValueError(
        "Required classes missing from HAM10000 metadata: "
        + str(missing_task_classes)
    )

print("Required classes: PASS")


# ============================================================
# 12. FINAL STEP-1 SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("STEP 1 AUDIT SUMMARY")
print("=" * 80)

print(
    "CUDA                         : PASS"
)

print(
    "Metadata                     : PASS"
)

print(
    "Required columns             : PASS"
)

print(
    "Duplicate image IDs          :",
    int(df["image_id"].duplicated().sum())
)

print(
    "Missing age                  :",
    missing_age
)

print(
    "Missing labels               :",
    missing_label
)

print(
    "Metadata IDs without images  :",
    len(missing_images)
)

print(
    "Images without metadata      :",
    len(unmatched_images)
)

print(
    "Task classes available       : PASS"
)

print("\n" + "=" * 80)
print("STEP 1 COMPLETE")
print("=" * 80)

BIASPRUNER — STEP 1
ENVIRONMENT + HAM10000 DATASET AUDIT

--------------------------------------------------------------------------------
1. ENVIRONMENT
--------------------------------------------------------------------------------
Python version : 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch version: 2.11.0+cu128
Torchvision    : 0.26.0+cu128
CUDA available : True
Device         : cuda
GPU count      : 2

GPU 0
  Name       : Tesla T4
  VRAM       : 14.56 GB
  Compute    : 7.5

GPU 1
  Name       : Tesla T4
  VRAM       : 14.56 GB
  Compute    : 7.5

Selected GPU: Tesla T4

--------------------------------------------------------------------------------
2. RUNTIME
--------------------------------------------------------------------------------
Working directory: /kaggle/working
Platform         : Linux-6.18.48+-x86_64-with-glibc2.39

--------------------------------------------------------------------------------
3. SEARCHING FOR HAM10000
-----------------------------

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear



--------------------------------------------------------------------------------
4. METADATA SCHEMA CHECK
--------------------------------------------------------------------------------
Required columns: PASS

--------------------------------------------------------------------------------
5. DATASET AUDIT
--------------------------------------------------------------------------------
Total metadata rows : 10015
Unique image IDs    : 10015
Duplicate image IDs : 0

Class distribution:
dx
akiec     327
bcc       514
bkl      1099
df        115
mel      1113
nv       6705
vasc      142
Name: count, dtype: int64

Missing values:
image_id     0
dx           0
age         57
dtype: int64

Missing age      : 57
Missing labels   : 0
Missing image ID : 0

--------------------------------------------------------------------------------
6. AGE DISTRIBUTION
--------------------------------------------------------------------------------
count    9958.000000
mean       51.863828
std        16.96

In [3]:
# ============================================================
# STEP 2 — REPRODUCIBLE LESION-LEVEL SPLIT
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold
from pathlib import Path
import numpy as np
import pandas as pd

print("=" * 80)
print("BIASPRUNER — STEP 2")
print("REPRODUCIBLE LESION-LEVEL SPLIT")
print("=" * 80)


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42

np.random.seed(SEED)

SPLIT_FILENAME = "ham10000_lesion_split_seed42.csv"

print("\nSeed:", SEED)


# ============================================================
# 2. COPY METADATA
# ============================================================

split_df = df.copy()

print("\nOriginal dataset shape:")
print(split_df.shape)


# ============================================================
# 3. CREATE AGE STRATIFICATION GROUP
# ============================================================

# IMPORTANT:
# We DO NOT delete the 57 missing-age samples.
#
# They are assigned to a separate "missing" group only
# for stratification.
#
# Later, missing-age samples will be excluded ONLY from
# age-based fairness calculations.

split_df["age_numeric"] = pd.to_numeric(
    split_df["age"],
    errors="coerce"
)

split_df["age_group"] = np.where(
    split_df["age_numeric"].isna(),
    "missing",
    np.where(
        split_df["age_numeric"] < 60,
        "<60",
        ">=60"
    )
)

print("\nAge-group distribution:")

print(
    split_df["age_group"]
    .value_counts(dropna=False)
)


# ============================================================
# 4. CREATE STRATIFICATION LABEL
# ============================================================

# Combine disease class and age group.
#
# Example:
#
# bkl_<60
# bkl_>=60
# bkl_missing
# mel_<60
# ...

split_df["stratify_group"] = (
    split_df["dx"].astype(str)
    + "__"
    + split_df["age_group"].astype(str)
)

print("\nNumber of stratification groups:")
print(
    split_df["stratify_group"].nunique()
)


# ============================================================
# 5. CHECK GROUP INFORMATION
# ============================================================

print("\nUnique lesions:")
print(
    split_df["lesion_id"].nunique()
)

print(
    "Total images:",
    len(split_df)
)

print(
    "Images per lesion — summary:"
)

print(
    split_df
    .groupby("lesion_id")
    .size()
    .describe()
)


# ============================================================
# 6. STRATIFIED GROUP K-FOLD
# ============================================================

print("\n" + "-" * 80)
print("Creating lesion-level folds...")
print("-" * 80)

sgkf = StratifiedGroupKFold(
    n_splits=10,
    shuffle=True,
    random_state=SEED
)

split_df["fold"] = -1

X = split_df["image_id"].values
y = split_df["stratify_group"].values
groups = split_df["lesion_id"].values

for fold_number, (_, fold_indices) in enumerate(
    sgkf.split(X, y, groups)
):

    split_df.loc[
        fold_indices,
        "fold"
    ] = fold_number


# ============================================================
# 7. VERIFY ALL ROWS GOT A FOLD
# ============================================================

if (split_df["fold"] == -1).any():

    raise RuntimeError(
        "Some images were not assigned to a fold."
    )

print("\nFold counts:")

print(
    split_df["fold"]
    .value_counts()
    .sort_index()
)


# ============================================================
# 8. ASSIGN TRAIN / VALIDATION / TEST
# ============================================================

# 10 folds:
#
# 0       -> TEST       (~10%)
# 1,2     -> VALIDATION (~20%)
# 3-9     -> TRAIN      (~70%)

split_df["split"] = np.select(
    [
        split_df["fold"] == 0,
        split_df["fold"].isin([1, 2])
    ],
    [
        "test",
        "validation"
    ],
    default="train"
)


# ============================================================
# 9. BASIC SPLIT SIZES
# ============================================================

print("\n" + "-" * 80)
print("SPLIT SIZES")
print("-" * 80)

split_counts = (
    split_df["split"]
    .value_counts()
    .reindex(
        ["train", "validation", "test"]
    )
)

print(split_counts)

print("\nPercentages:")

print(
    (split_counts / len(split_df) * 100)
    .round(2)
)


# ============================================================
# 10. VERIFY LESION LEAKAGE
# ============================================================

print("\n" + "-" * 80)
print("LESION-LEVEL LEAKAGE CHECK")
print("-" * 80)

lesion_split_counts = (
    split_df
    .groupby("lesion_id")["split"]
    .nunique()
)

leaking_lesions = (
    lesion_split_counts[
        lesion_split_counts > 1
    ]
)

print(
    "Lesions appearing in multiple splits:",
    len(leaking_lesions)
)

if len(leaking_lesions) > 0:

    print(
        "\nERROR: Lesion leakage detected!"
    )

    print(
        leaking_lesions.head(20)
    )

    raise RuntimeError(
        "Lesion-level data leakage detected."
    )

print(
    "Lesion leakage check: PASS"
)


# ============================================================
# 11. VERIFY IMAGE-ID LEAKAGE
# ============================================================

image_split_counts = (
    split_df
    .groupby("image_id")["split"]
    .nunique()
)

duplicate_split_images = (
    image_split_counts[
        image_split_counts > 1
    ]
)

print(
    "Images appearing in multiple splits:",
    len(duplicate_split_images)
)

if len(duplicate_split_images) > 0:

    raise RuntimeError(
        "Image-level leakage detected."
    )

print(
    "Image leakage check: PASS"
)


# ============================================================
# 12. CLASS DISTRIBUTION BY SPLIT
# ============================================================

print("\n" + "-" * 80)
print("CLASS DISTRIBUTION BY SPLIT")
print("-" * 80)

class_split_table = pd.crosstab(
    split_df["dx"],
    split_df["split"]
).reindex(
    columns=["train", "validation", "test"]
)

print(class_split_table)


# ============================================================
# 13. AGE DISTRIBUTION BY SPLIT
# ============================================================

print("\n" + "-" * 80)
print("AGE-GROUP DISTRIBUTION BY SPLIT")
print("-" * 80)

age_split_table = pd.crosstab(
    split_df["age_group"],
    split_df["split"]
).reindex(
    index=["<60", ">=60", "missing"],
    columns=["train", "validation", "test"]
)

print(age_split_table)


# ============================================================
# 14. REQUIRED TASK CLASS DISTRIBUTION
# ============================================================

print("\n" + "-" * 80)
print("REQUIRED TASK CLASSES")
print("-" * 80)

TASK1_CLASSES = ["bkl", "mel"]
TASK2_CLASSES = ["bcc", "akiec"]

task_classes = TASK1_CLASSES + TASK2_CLASSES

task_split_df = split_df[
    split_df["dx"].isin(task_classes)
].copy()

task_distribution = pd.crosstab(
    task_split_df["dx"],
    task_split_df["split"]
).reindex(
    index=task_classes,
    columns=["train", "validation", "test"],
    fill_value=0
)

print(task_distribution)


# ============================================================
# 15. VERIFY EACH TASK HAS ALL CLASSES
# ============================================================

for task_name, classes in [
    ("Task 1", TASK1_CLASSES),
    ("Task 2", TASK2_CLASSES)
]:

    print(f"\n{task_name}")

    for split_name in [
        "train",
        "validation",
        "test"
    ]:

        subset = task_split_df[
            task_split_df["split"] == split_name
        ]

        present_classes = set(
            subset["dx"]
        )

        missing = [
            c for c in classes
            if c not in present_classes
        ]

        print(
            f"  {split_name}:",
            "PASS" if not missing
            else f"FAIL — missing {missing}"
        )

        if missing:

            raise RuntimeError(
                f"{task_name} {split_name} "
                f"is missing classes: {missing}"
            )


# ============================================================
# 16. SAVE CLEAN SPLIT FILE
# ============================================================

# Keep only columns needed for the experiment.
#
# age_numeric and age_group are retained because they are
# useful for later fairness analysis.

final_columns = [
    "lesion_id",
    "image_id",
    "dx",
    "dx_type",
    "age",
    "age_numeric",
    "age_group",
    "sex",
    "localization",
    "fold",
    "split"
]

split_output = split_df[
    final_columns
].copy()

output_path = Path(
    "/kaggle/working"
) / SPLIT_FILENAME

split_output.to_csv(
    output_path,
    index=False
)

print("\n" + "-" * 80)
print("SPLIT SAVED")
print("-" * 80)

print(output_path)

print(
    "File exists:",
    output_path.exists()
)

print(
    "File size:",
    round(
        output_path.stat().st_size / 1024,
        2
    ),
    "KB"
)


# ============================================================
# 17. FINAL SANITY CHECK — RELOAD FILE
# ============================================================

reloaded = pd.read_csv(
    output_path
)

print("\nReloaded split shape:")
print(reloaded.shape)

if len(reloaded) != len(split_df):

    raise RuntimeError(
        "Saved split does not contain all rows."
    )

print(
    "Reload verification: PASS"
)


# ============================================================
# 18. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("STEP 2 SUMMARY")
print("=" * 80)

print(
    "Total images       :",
    len(split_output)
)

print(
    "Unique lesions      :",
    split_output["lesion_id"].nunique()
)

print(
    "Train images        :",
    (split_output["split"] == "train").sum()
)

print(
    "Validation images   :",
    (split_output["split"] == "validation").sum()
)

print(
    "Test images         :",
    (split_output["split"] == "test").sum()
)

print(
    "Lesion leakage      : 0"
)

print(
    "Image leakage       : 0"
)

print(
    "Split file          :",
    output_path
)

print("\n" + "=" * 80)
print("STEP 2 COMPLETE")
print("=" * 80)

BIASPRUNER — STEP 2
REPRODUCIBLE LESION-LEVEL SPLIT

Seed: 42

Original dataset shape:
(10015, 7)

Age-group distribution:
age_group
<60        6356
>=60       3602
missing      57
Name: count, dtype: int64

Number of stratification groups:
17

Unique lesions:
7470
Total images: 10015
Images per lesion — summary:
count    7470.000000
mean        1.340696
std         0.633970
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         6.000000
dtype: float64

--------------------------------------------------------------------------------
Creating lesion-level folds...
--------------------------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:1023: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=10.
  warnings.warn(



Fold counts:
fold
0    1015
1     978
2    1003
3     989
4    1008
5    1010
6    1014
7    1000
8     980
9    1018
Name: count, dtype: int64

--------------------------------------------------------------------------------
SPLIT SIZES
--------------------------------------------------------------------------------
split
train         7019
validation    1981
test          1015
Name: count, dtype: int64

Percentages:
split
train         70.08
validation    19.78
test          10.13
Name: count, dtype: float64

--------------------------------------------------------------------------------
LESION-LEVEL LEAKAGE CHECK
--------------------------------------------------------------------------------
Lesions appearing in multiple splits: 0
Lesion leakage check: PASS
Images appearing in multiple splits: 0
Image leakage check: PASS

--------------------------------------------------------------------------------
CLASS DISTRIBUTION BY SPLIT
---------------------------------------------------

In [4]:
# ============================================================================
# BIASPRUNER — STEP 3
# TASK 1 / TASK 2 DATASETS + DATALOADERS
# ============================================================================

print("=" * 80)
print("BIASPRUNER — STEP 3 TASK 1 / TASK 2 DATALOADERS")
print("=" * 80)

import os
import json
import glob
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# ----------------------------------------------------------------------------
# 1. REPRODUCIBILITY
# ----------------------------------------------------------------------------

SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)

# ----------------------------------------------------------------------------
# 2. CONFIGURATION
# ----------------------------------------------------------------------------

IMAGE_SIZE = 224
BATCH_SIZE = 32

NUM_WORKERS = min(2, os.cpu_count() or 1)
PIN_MEMORY = torch.cuda.is_available()

SPLIT_FILE = "/kaggle/working/ham10000_lesion_split_seed42.csv"

print("Image size :", IMAGE_SIZE)
print("Batch size :", BATCH_SIZE)
print("Workers    :", NUM_WORKERS)
print("Pin memory :", PIN_MEMORY)

# ----------------------------------------------------------------------------
# 3. LOAD EXACT SPLIT CREATED IN STEP 2
# ----------------------------------------------------------------------------

split_df = pd.read_csv(SPLIT_FILE)

required_columns = [
    "lesion_id",
    "image_id",
    "dx",
    "age",
    "age_numeric",
    "age_group",
    "split"
]

missing_columns = [
    c for c in required_columns
    if c not in split_df.columns
]

if missing_columns:
    raise RuntimeError(
        f"Required columns missing from split file: {missing_columns}"
    )

print("\nLoaded split:")
print("Shape:", split_df.shape)

print("\nSplit distribution:")
print(split_df["split"].value_counts())

# ----------------------------------------------------------------------------
# 4. DISCOVER HAM10000 IMAGE FILES
# ----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("DISCOVERING IMAGE FILES")
print("-" * 80)

# Search under Kaggle input.
all_image_files = glob.glob(
    "/kaggle/input/**/*.jpg",
    recursive=True
)

print("Total .jpg files found:", len(all_image_files))

if len(all_image_files) == 0:
    raise FileNotFoundError(
        "No .jpg images found under /kaggle/input/"
    )

# Build image_id -> filepath mapping.
# HAM10000 filenames are like:
# ISIC_0027419.jpg
image_path_map = {}

duplicate_image_ids = []

for path in all_image_files:

    filename = os.path.basename(path)

    if not filename.lower().endswith(".jpg"):
        continue

    image_id = os.path.splitext(filename)[0]

    if image_id in image_path_map:
        duplicate_image_ids.append(image_id)
    else:
        image_path_map[image_id] = path

print("Unique image IDs discovered:", len(image_path_map))
print("Duplicate image IDs:", len(set(duplicate_image_ids)))

# ----------------------------------------------------------------------------
# 5. VERIFY EVERY SPLIT IMAGE HAS A FILE
# ----------------------------------------------------------------------------

split_image_ids = set(split_df["image_id"].astype(str))
disk_image_ids = set(image_path_map.keys())

missing_on_disk = split_image_ids - disk_image_ids
extra_on_disk = disk_image_ids - split_image_ids

print("Image IDs in split :", len(split_image_ids))
print("Image IDs on disk :", len(disk_image_ids))
print("Missing on disk   :", len(missing_on_disk))
print("Extra on disk     :", len(extra_on_disk))

if missing_on_disk:
    print("\nMissing image IDs:")
    print(sorted(list(missing_on_disk))[:20])
    raise RuntimeError(
        "Some images in the split cannot be found on disk."
    )

print("Image correspondence: PASS")

# ----------------------------------------------------------------------------
# 6. ADD IMAGE PATH TO SPLIT DATAFRAME
# ----------------------------------------------------------------------------

split_df["image_path"] = split_df["image_id"].astype(str).map(
    image_path_map
)

if split_df["image_path"].isna().any():
    raise RuntimeError(
        "Some rows have no resolved image path."
    )

print("Image path mapping: PASS")

# ----------------------------------------------------------------------------
# 7. DEFINE THE TWO REQUIRED TASKS
# ----------------------------------------------------------------------------

TASK1_CLASSES = ["bkl", "mel"]
TASK2_CLASSES = ["bcc", "akiec"]

TASK1_LABEL_MAP = {
    "bkl": 0,
    "mel": 1
}

TASK2_LABEL_MAP = {
    "bcc": 0,
    "akiec": 1
}

print("\nTask 1:")
print("  bkl  -> 0")
print("  mel  -> 1")

print("\nTask 2:")
print("  bcc  -> 0")
print("  akiec -> 1")

# ----------------------------------------------------------------------------
# 8. FILTER DATA FOR EACH TASK
# ----------------------------------------------------------------------------

task1_df = split_df[
    split_df["dx"].isin(TASK1_CLASSES)
].copy()

task2_df = split_df[
    split_df["dx"].isin(TASK2_CLASSES)
].copy()

task1_df["label"] = task1_df["dx"].map(TASK1_LABEL_MAP)
task2_df["label"] = task2_df["dx"].map(TASK2_LABEL_MAP)

# Sanity checks
assert task1_df["label"].notna().all()
assert task2_df["label"].notna().all()

assert set(task1_df["label"].unique()) == {0, 1}
assert set(task2_df["label"].unique()) == {0, 1}

print("\nTask 1 total images:", len(task1_df))
print("Task 2 total images:", len(task2_df))

print("\nTask 1 class distribution:")
print(
    task1_df["dx"].value_counts()
)

print("\nTask 2 class distribution:")
print(
    task2_df["dx"].value_counts()
)

# ----------------------------------------------------------------------------
# 9. VERIFY EACH TASK HAS TRAIN / VALIDATION / TEST
# ----------------------------------------------------------------------------

for task_name, task_df, classes in [
    ("Task 1", task1_df, TASK1_CLASSES),
    ("Task 2", task2_df, TASK2_CLASSES)
]:

    print("\n" + task_name)

    for split_name in ["train", "validation", "test"]:

        subset = task_df[
            task_df["split"] == split_name
        ]

        present_classes = set(subset["dx"].unique())

        print(
            f"  {split_name:10s}: "
            f"{len(subset):4d} images | "
            f"classes = {sorted(present_classes)}"
        )

        if present_classes != set(classes):
            raise RuntimeError(
                f"{task_name} {split_name} does not contain "
                f"both required classes."
            )

print("\nTask split verification: PASS")

# ----------------------------------------------------------------------------
# 10. DATASET CLASS
# ----------------------------------------------------------------------------

class HAM10000TaskDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None
    ):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):

        row = self.df.iloc[index]

        image_path = row["image_path"]
        label = int(row["label"])

        # Open image
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, label


# ----------------------------------------------------------------------------
# 11. TEMPORARY TRANSFORM FOR TRAINING-ONLY NORMALIZATION CALCULATION
# ----------------------------------------------------------------------------

# IMPORTANT:
# No validation/test image is used to calculate mean/std.
# No random augmentation is used here.

normalization_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor()
])

# ----------------------------------------------------------------------------
# 12. CREATE A COMBINED TRAINING DATAFRAME
# ----------------------------------------------------------------------------

# Normalization statistics are computed ONLY from images belonging
# to the training split.

normalization_train_df = split_df[
    split_df["split"] == "train"
].copy()

print("\nTraining images used for normalization:", len(normalization_train_df))

normalization_dataset = HAM10000TaskDataset(
    normalization_train_df.assign(
        label=0
    ),
    transform=normalization_transform
)

normalization_loader = DataLoader(
    normalization_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

# ----------------------------------------------------------------------------
# 13. COMPUTE TRAINING-ONLY MEAN AND STD
# ----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("COMPUTING TRAINING-ONLY NORMALIZATION")
print("-" * 80)

channel_sum = torch.zeros(3)
channel_sum_squared = torch.zeros(3)
total_pixels = 0

with torch.no_grad():

    for batch_images, _ in normalization_loader:

        # batch_images:
        # [B, 3, H, W]

        batch_size_current = batch_images.size(0)

        pixels = batch_size_current * IMAGE_SIZE * IMAGE_SIZE

        channel_sum += batch_images.sum(
            dim=(0, 2, 3)
        )

        channel_sum_squared += (
            batch_images ** 2
        ).sum(
            dim=(0, 2, 3)
        )

        total_pixels += pixels

mean = channel_sum / total_pixels

variance = (
    channel_sum_squared / total_pixels
) - mean ** 2

# Numerical protection
variance = torch.clamp(
    variance,
    min=0.0
)

std = torch.sqrt(variance)

print("Training mean:", mean.tolist())
print("Training std :", std.tolist())

# ----------------------------------------------------------------------------
# 14. SAVE NORMALIZATION STATISTICS
# ----------------------------------------------------------------------------

normalization_stats = {
    "seed": SEED,
    "image_size": IMAGE_SIZE,
    "mean": mean.tolist(),
    "std": std.tolist(),
    "source": "TRAINING SPLIT ONLY",
    "num_training_images": len(normalization_train_df)
}

NORMALIZATION_FILE = (
    "/kaggle/working/ham10000_normalization_seed42.json"
)

with open(
    NORMALIZATION_FILE,
    "w"
) as f:
    json.dump(
        normalization_stats,
        f,
        indent=4
    )

print("\nNormalization statistics saved:")
print(NORMALIZATION_FILE)

# ----------------------------------------------------------------------------
# 15. DEFINE FINAL TRANSFORMS
# ----------------------------------------------------------------------------

# TRAIN:
# Resize + Random Horizontal Flip + ToTensor + TRAINING-ONLY normalization

train_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.RandomHorizontalFlip(
        p=0.5
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean.tolist(),
        std=std.tolist()
    )
])

# VALIDATION:
# No random augmentation

validation_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean.tolist(),
        std=std.tolist()
    )
])

# TEST:
# No random augmentation

test_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean.tolist(),
        std=std.tolist()
    )
])

# ----------------------------------------------------------------------------
# 16. HELPER FUNCTION FOR TASK DATALOADERS
# ----------------------------------------------------------------------------

def create_task_dataloaders(
    task_df,
    task_name
):

    train_df = task_df[
        task_df["split"] == "train"
    ].copy()

    validation_df = task_df[
        task_df["split"] == "validation"
    ].copy()

    test_df = task_df[
        task_df["split"] == "test"
    ].copy()

    train_dataset = HAM10000TaskDataset(
        train_df,
        transform=train_transform
    )

    validation_dataset = HAM10000TaskDataset(
        validation_df,
        transform=validation_transform
    )

    test_dataset = HAM10000TaskDataset(
        test_df,
        transform=test_transform
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False
    )

    validation_loader = DataLoader(
        validation_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False
    )

    print("\n" + task_name)
    print("-" * 40)
    print("Train      :", len(train_dataset))
    print("Validation :", len(validation_dataset))
    print("Test       :", len(test_dataset))

    return (
        train_loader,
        validation_loader,
        test_loader
    )


# ----------------------------------------------------------------------------
# 17. CREATE TASK 1 DATALOADERS
# ----------------------------------------------------------------------------

(
    task1_train_loader,
    task1_val_loader,
    task1_test_loader
) = create_task_dataloaders(
    task1_df,
    "TASK 1 — BKL vs MEL"
)

# ----------------------------------------------------------------------------
# 18. CREATE TASK 2 DATALOADERS
# ----------------------------------------------------------------------------

(
    task2_train_loader,
    task2_val_loader,
    task2_test_loader
) = create_task_dataloaders(
    task2_df,
    "TASK 2 — BCC vs AKIEC"
)

# ----------------------------------------------------------------------------
# 19. BATCH SHAPE SANITY CHECK
# ----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("BATCH SANITY CHECK")
print("-" * 80)

task1_images, task1_labels = next(
    iter(task1_train_loader)
)

task2_images, task2_labels = next(
    iter(task2_train_loader)
)

print("\nTask 1:")
print("Images shape :", task1_images.shape)
print("Labels shape :", task1_labels.shape)
print("Labels       :", task1_labels[:10].tolist())

print("\nTask 2:")
print("Images shape :", task2_images.shape)
print("Labels shape :", task2_labels.shape)
print("Labels       :", task2_labels[:10].tolist())

# Expected:
# Images = [32, 3, 224, 224]
# Labels = [32]

assert task1_images.ndim == 4
assert task1_images.shape[1:] == (3, 224, 224)

assert task2_images.ndim == 4
assert task2_images.shape[1:] == (3, 224, 224)

assert task1_labels.ndim == 1
assert task2_labels.ndim == 1

assert set(task1_labels.tolist()).issubset({0, 1})
assert set(task2_labels.tolist()).issubset({0, 1})

print("\nBatch shape verification: PASS")

# ----------------------------------------------------------------------------
# 20. NORMALIZATION SANITY CHECK
# ----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("NORMALIZATION SANITY CHECK")
print("-" * 80)

print(
    "Task 1 batch mean:",
    task1_images.mean(
        dim=(0, 2, 3)
    ).tolist()
)

print(
    "Task 1 batch std :",
    task1_images.std(
        dim=(0, 2, 3)
    ).tolist()
)

print(
    "Task 2 batch mean:",
    task2_images.mean(
        dim=(0, 2, 3)
    ).tolist()
)

print(
    "Task 2 batch std :",
    task2_images.std(
        dim=(0, 2, 3)
    ).tolist()
)

# ----------------------------------------------------------------------------
# 21. SAVE TASK METADATA
# ----------------------------------------------------------------------------

task_summary = {
    "seed": SEED,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,

    "task1": {
        "classes": TASK1_CLASSES,
        "label_map": TASK1_LABEL_MAP,
        "train": int((task1_df["split"] == "train").sum()),
        "validation": int(
            (task1_df["split"] == "validation").sum()
        ),
        "test": int((task1_df["split"] == "test").sum())
    },

    "task2": {
        "classes": TASK2_CLASSES,
        "label_map": TASK2_LABEL_MAP,
        "train": int((task2_df["split"] == "train").sum()),
        "validation": int(
            (task2_df["split"] == "validation").sum()
        ),
        "test": int((task2_df["split"] == "test").sum())
    },

    "normalization": {
        "mean": mean.tolist(),
        "std": std.tolist(),
        "computed_from": "training split only"
    }
}

TASK_SUMMARY_FILE = (
    "/kaggle/working/ham10000_task_dataloader_summary.json"
)

with open(
    TASK_SUMMARY_FILE,
    "w"
) as f:
    json.dump(
        task_summary,
        f,
        indent=4
    )

print("\nTask summary saved:")
print(TASK_SUMMARY_FILE)

# ----------------------------------------------------------------------------
# FINAL STEP 3 SUMMARY
# ----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 3 SUMMARY")
print("=" * 80)

print("Task 1: BKL + MEL")
print("  Train      :", len(task1_df[task1_df["split"] == "train"]))
print("  Validation :", len(task1_df[task1_df["split"] == "validation"]))
print("  Test       :", len(task1_df[task1_df["split"] == "test"]))

print("\nTask 2: BCC + AKIEC")
print("  Train      :", len(task2_df[task2_df["split"] == "train"]))
print("  Validation :", len(task2_df[task2_df["split"] == "validation"]))
print("  Test       :", len(task2_df[task2_df["split"] == "test"]))

print("\nNormalization:")
print("  Mean:", mean.tolist())
print("  Std :", std.tolist())
print("  Source: TRAINING SPLIT ONLY")

print("\nTransforms:")
print("  Train      : Resize + RandomHorizontalFlip + Normalize")
print("  Validation : Resize + Normalize")
print("  Test       : Resize + Normalize")

print("\nBatch size:", BATCH_SIZE)
print("Image size:", IMAGE_SIZE)

print("\n" + "=" * 80)
print("STEP 3 COMPLETE")
print("=" * 80)

BIASPRUNER — STEP 3 TASK 1 / TASK 2 DATALOADERS
Seed: 42
Image size : 224
Batch size : 32
Workers    : 2
Pin memory : True

Loaded split:
Shape: (10015, 11)

Split distribution:
split
train         7019
validation    1981
test          1015
Name: count, dtype: int64

--------------------------------------------------------------------------------
DISCOVERING IMAGE FILES
--------------------------------------------------------------------------------
Total .jpg files found: 20030
Unique image IDs discovered: 10015
Duplicate image IDs: 10015
Image IDs in split : 10015
Image IDs on disk : 10015
Missing on disk   : 0
Extra on disk     : 0
Image correspondence: PASS
Image path mapping: PASS

Task 1:
  bkl  -> 0
  mel  -> 1

Task 2:
  bcc  -> 0
  akiec -> 1

Task 1 total images: 2212
Task 2 total images: 841

Task 1 class distribution:
dx
mel    1113
bkl    1099
Name: count, dtype: int64

Task 2 class distribution:
dx
bcc      514
akiec    327
Name: count, dtype: int64

Task 1
  train     : 

In [7]:
# ============================================================================
# BIASPRUNER — STEP 4
# TASK 1 GENERALIZED CROSS ENTROPY (GCE) TRAINING
# ============================================================================

print("=" * 80)
print("BIASPRUNER — STEP 4 TASK 1 GCE TRAINING")
print("=" * 80)

import os
import json
import time
import copy
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet50

# ----------------------------------------------------------------------------
# 1. REPRODUCIBILITY
# ----------------------------------------------------------------------------

SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Reproducibility settings
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ----------------------------------------------------------------------------
# 2. DEVICE
# ----------------------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ----------------------------------------------------------------------------
# 3. GCE CONFIGURATION
# ----------------------------------------------------------------------------

GCE_Q = 0.7

MAX_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 5
MIN_DELTA = 1e-4

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.0

NUM_CLASSES = 2

print("\nGCE configuration:")
print("q                    :", GCE_Q)
print("Maximum epochs       :", MAX_EPOCHS)
print("Early stopping       :", EARLY_STOPPING_PATIENCE)
print("Learning rate        :", LEARNING_RATE)
print("Weight decay         :", WEIGHT_DECAY)
print("Number of classes    :", NUM_CLASSES)

# ----------------------------------------------------------------------------
# 4. VERIFY REQUIRED VARIABLES FROM STEP 3
# ----------------------------------------------------------------------------

required_variables = [
    "task1_train_loader",
    "task1_val_loader",
    "task1_test_loader",
    "task1_df"
]

for variable_name in required_variables:

    if variable_name not in globals():
        raise RuntimeError(
            f"{variable_name} is missing. "
            "Please run Steps 1–3 first."
        )

print("\nStep 3 variables: PASS")

# ----------------------------------------------------------------------------
# 5. GCE LOSS
# ----------------------------------------------------------------------------

def generalized_cross_entropy(
    logits,
    targets,
    q=0.7
):
    """
    Generalized Cross Entropy:

        L_GCE = (1 - p_y^q) / q

    where:
        p_y = predicted probability assigned to the true class.

    q -> 0 approaches CE.
    q = 0.7 is the BiasPruner setting used here.
    """

    probabilities = torch.softmax(
        logits,
        dim=1
    )

    true_class_probability = probabilities.gather(
        1,
        targets.unsqueeze(1)
    ).squeeze(1)

    # Numerical protection
    true_class_probability = torch.clamp(
        true_class_probability,
        min=1e-7,
        max=1.0
    )

    loss = (
        1.0
        - true_class_probability.pow(q)
    ) / q

    return loss.mean()


# ----------------------------------------------------------------------------
# 6. VALIDATION METRICS
# ----------------------------------------------------------------------------

def compute_binary_age_metrics(
    predictions,
    targets,
    dataframe
):
    """
    Computes:
      - Accuracy
      - TPR for age < 60
      - TPR for age >= 60
      - EOD

    Missing-age samples are excluded ONLY from age fairness metrics.

    EOD:
        TPR(<60) - TPR(>=60)

    We also return |EOD|.
    """

    predictions = np.asarray(predictions)
    targets = np.asarray(targets)

    accuracy = float(
        (predictions == targets).mean()
    )

    evaluation_df = dataframe.reset_index(drop=True).copy()

    if len(evaluation_df) != len(predictions):
        raise RuntimeError(
            "Prediction count does not match dataframe length."
        )

    evaluation_df["prediction"] = predictions
    evaluation_df["target"] = targets

    # Classification accuracy includes all samples.
    accuracy_lt60 = np.nan
    accuracy_ge60 = np.nan

    # Age-specific classification accuracy
    age_lt60_mask = (
        evaluation_df["age_group"] == "<60"
    )

    age_ge60_mask = (
        evaluation_df["age_group"] == ">=60"
    )

    if age_lt60_mask.sum() > 0:
        accuracy_lt60 = float(
            (
                evaluation_df.loc[
                    age_lt60_mask,
                    "prediction"
                ]
                ==
                evaluation_df.loc[
                    age_lt60_mask,
                    "target"
                ]
            ).mean()
        )

    if age_ge60_mask.sum() > 0:
        accuracy_ge60 = float(
            (
                evaluation_df.loc[
                    age_ge60_mask,
                    "prediction"
                ]
                ==
                evaluation_df.loc[
                    age_ge60_mask,
                    "target"
                ]
            ).mean()
        )

    # ------------------------------------------------------------------------
    # TPR / EOD
    #
    # For binary classification, positive class = label 1 = MEL.
    # ------------------------------------------------------------------------

    def calculate_tpr(mask):

        group_targets = evaluation_df.loc[
            mask,
            "target"
        ].values

        group_predictions = evaluation_df.loc[
            mask,
            "prediction"
        ].values

        positive_mask = (
            group_targets == 1
        )

        positive_count = positive_mask.sum()

        if positive_count == 0:
            return np.nan

        true_positives = (
            (group_predictions == 1)
            &
            (group_targets == 1)
        ).sum()

        return float(
            true_positives / positive_count
        )

    tpr_lt60 = calculate_tpr(
        age_lt60_mask
    )

    tpr_ge60 = calculate_tpr(
        age_ge60_mask
    )

    if (
        np.isnan(tpr_lt60)
        or np.isnan(tpr_ge60)
    ):
        eod = np.nan
    else:
        eod = tpr_lt60 - tpr_ge60

    abs_eod = (
        abs(eod)
        if not np.isnan(eod)
        else np.nan
    )

    return {
        "accuracy": accuracy,
        "accuracy_lt60": accuracy_lt60,
        "accuracy_ge60": accuracy_ge60,
        "tpr_lt60": tpr_lt60,
        "tpr_ge60": tpr_ge60,
        "eod": eod,
        "abs_eod": abs_eod
    }


# ----------------------------------------------------------------------------
# 7. GET TASK 1 VALIDATION DATAFRAME IN EXACT LOADER ORDER
# ----------------------------------------------------------------------------

task1_val_df = task1_df[
    task1_df["split"] == "validation"
].copy()

task1_val_df = task1_val_df.reset_index(
    drop=True
)

print("\nTask 1 validation samples:", len(task1_val_df))

# ----------------------------------------------------------------------------
# 8. CREATE RESNET-50
# ----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("CREATING RESNET-50")
print("-" * 80)

# The paper's HAM configuration uses a ResNet-50 backbone.
# We explicitly use random initialization here rather than ImageNet
# pretrained weights.

model = resnet50(
    weights=None
)

# Replace the original 1000-class ImageNet classifier
# with a 2-class classifier.

in_features = model.fc.in_features

model.fc = nn.Linear(
    in_features,
    NUM_CLASSES
)

model = model.to(device)

print(model.fc)

# ----------------------------------------------------------------------------
# 9. COUNT PARAMETERS
# ----------------------------------------------------------------------------

total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nTotal parameters    :", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")

# ----------------------------------------------------------------------------
# 10. OPTIMIZER
# ----------------------------------------------------------------------------

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# ----------------------------------------------------------------------------
# 11. OUTPUT DIRECTORY
# ----------------------------------------------------------------------------

OUTPUT_DIR = "/kaggle/working/BiasPruner_HAM10000"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

GCE_CHECKPOINT = os.path.join(
    OUTPUT_DIR,
    "task1_gce_best.pt"
)

GCE_HISTORY_FILE = os.path.join(
    OUTPUT_DIR,
    "task1_gce_history.csv"
)

print("\nOutput directory:")
print(OUTPUT_DIR)

# ----------------------------------------------------------------------------
# 12. EVALUATION FUNCTION
# ----------------------------------------------------------------------------

def evaluate_gce_model(
    model,
    loader,
    dataframe
):

    model.eval()

    all_predictions = []
    all_targets = []
    all_losses = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            logits = model(images)

            loss = generalized_cross_entropy(
                logits,
                labels,
                q=GCE_Q
            )

            probabilities = torch.softmax(
                logits,
                dim=1
            )

            predictions = torch.argmax(
                probabilities,
                dim=1
            )

            all_losses.append(
                loss.item()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_targets.extend(
                labels.cpu().numpy()
            )

    predictions = np.asarray(
        all_predictions
    )

    targets = np.asarray(
        all_targets
    )

    metrics = compute_binary_age_metrics(
        predictions,
        targets,
        dataframe
    )

    metrics["loss"] = float(
        np.mean(all_losses)
    )

    return metrics


# ----------------------------------------------------------------------------
# 13. TRAINING LOOP
# ----------------------------------------------------------------------------

history = []

best_val_accuracy = -np.inf
best_val_abs_eod = np.inf
best_epoch = None

best_state_dict = None

epochs_without_improvement = 0

training_start = time.time()

print("\n" + "=" * 80)
print("STARTING TASK 1 GCE TRAINING")
print("=" * 80)

for epoch in range(1, MAX_EPOCHS + 1):

    epoch_start = time.time()

    # ========================================================================
    # TRAIN
    # ========================================================================

    model.train()

    train_losses = []

    train_predictions = []
    train_targets = []

    for images, labels in task1_train_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(images)

        loss = generalized_cross_entropy(
            logits,
            labels,
            q=GCE_Q
        )

        loss.backward()

        optimizer.step()

        train_losses.append(
            loss.item()
        )

        predictions = torch.argmax(
            logits,
            dim=1
        )

        train_predictions.extend(
            predictions.detach().cpu().numpy()
        )

        train_targets.extend(
            labels.detach().cpu().numpy()
        )

    train_predictions = np.asarray(
        train_predictions
    )

    train_targets = np.asarray(
        train_targets
    )

    train_accuracy = float(
        (
            train_predictions
            ==
            train_targets
        ).mean()
    )

    train_loss = float(
        np.mean(train_losses)
    )

    # ========================================================================
    # VALIDATION
    # ========================================================================

    val_metrics = evaluate_gce_model(
        model,
        task1_val_loader,
        task1_val_df
    )

    val_loss = val_metrics["loss"]
    val_accuracy = val_metrics["accuracy"]
    val_eod = val_metrics["eod"]
    val_abs_eod = val_metrics["abs_eod"]

    epoch_time = time.time() - epoch_start

    # ========================================================================
    # CHECKPOINT DECISION
    # ========================================================================

    improved = False

    # Primary criterion:
    # Higher validation accuracy.
    #
    # Tie-break:
    # Lower absolute EOD.

    if val_accuracy > (
        best_val_accuracy + MIN_DELTA
    ):

        improved = True

    elif (
        abs(
            val_accuracy
            - best_val_accuracy
        ) <= MIN_DELTA
        and not np.isnan(val_abs_eod)
        and val_abs_eod < best_val_abs_eod
    ):

        improved = True

    if improved:

        best_val_accuracy = val_accuracy

        if np.isnan(val_abs_eod):
            best_val_abs_eod = np.inf
        else:
            best_val_abs_eod = val_abs_eod

        best_epoch = epoch

        best_state_dict = copy.deepcopy(
            model.state_dict()
        )

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_accuracy": val_accuracy,
                "val_eod": val_eod,
                "val_abs_eod": val_abs_eod,
                "q": GCE_Q,
                "learning_rate": LEARNING_RATE,
                "seed": SEED,
                "task": "Task 1 — BKL vs MEL"
            },
            GCE_CHECKPOINT
        )

        epochs_without_improvement = 0

    else:

        epochs_without_improvement += 1

    # ========================================================================
    # SAVE HISTORY
    # ========================================================================

    history_row = {
        "epoch": epoch,

        "train_loss": train_loss,
        "train_accuracy": train_accuracy,

        "val_loss": val_loss,
        "val_accuracy": val_accuracy,

        "val_accuracy_lt60":
            val_metrics["accuracy_lt60"],

        "val_accuracy_ge60":
            val_metrics["accuracy_ge60"],

        "val_tpr_lt60":
            val_metrics["tpr_lt60"],

        "val_tpr_ge60":
            val_metrics["tpr_ge60"],

        "val_eod": val_eod,
        "val_abs_eod": val_abs_eod,

        "best_epoch": best_epoch,
        "epochs_without_improvement":
            epochs_without_improvement,

        "epoch_time_seconds":
            epoch_time
    }

    history.append(
        history_row
    )

    history_df = pd.DataFrame(
        history
    )

    history_df.to_csv(
        GCE_HISTORY_FILE,
        index=False
    )

    # ========================================================================
    # PRINT
    # ========================================================================

    print(
        f"Epoch {epoch:03d}/{MAX_EPOCHS} | "
        f"Train Loss {train_loss:.4f} | "
        f"Train Acc {train_accuracy:.4f} | "
        f"Val Loss {val_loss:.4f} | "
        f"Val Acc {val_accuracy:.4f} | "
        f"Val EOD {val_eod:.4f} | "
        f"|EOD| {val_abs_eod:.4f} | "
        f"{epoch_time:.1f}s"
    )

    # ========================================================================
    # EARLY STOPPING
    # ========================================================================

    if epochs_without_improvement >= (
        EARLY_STOPPING_PATIENCE
    ):

        print(
            "\nEarly stopping triggered."
        )

        print(
            "Best epoch:",
            best_epoch
        )

        break


training_time = (
    time.time() - training_start
)

# ----------------------------------------------------------------------------
# 14. RESTORE BEST MODEL
# ----------------------------------------------------------------------------

if best_state_dict is None:

    raise RuntimeError(
        "No valid checkpoint was produced."
    )

model.load_state_dict(
    best_state_dict
)

print("\n" + "=" * 80)
print("BEST TASK 1 GCE MODEL")
print("=" * 80)

print("Best epoch       :", best_epoch)
print("Best val accuracy:", best_val_accuracy)
print("Best |EOD|       :", best_val_abs_eod)
print(
    "Training time    :",
    f"{training_time / 60:.2f} minutes"
)

# ----------------------------------------------------------------------------
# 15. LOAD CHECKPOINT VERIFICATION
# ----------------------------------------------------------------------------

checkpoint = torch.load(
    GCE_CHECKPOINT,
    map_location=device
)

print("\nCheckpoint verification:")
print("Checkpoint epoch     :", checkpoint["epoch"])
print(
    "Checkpoint val acc  :",
    checkpoint["val_accuracy"]
)
print(
    "Checkpoint val EOD  :",
    checkpoint["val_eod"]
)
print(
    "Checkpoint |EOD|    :",
    checkpoint["val_abs_eod"]
)

# ----------------------------------------------------------------------------
# 16. FINAL VALIDATION EVALUATION OF BEST CHECKPOINT
# ----------------------------------------------------------------------------

best_val_metrics = evaluate_gce_model(
    model,
    task1_val_loader,
    task1_val_df
)

print("\n" + "-" * 80)
print("BEST CHECKPOINT — VALIDATION RESULTS")
print("-" * 80)

for key, value in best_val_metrics.items():

    if isinstance(value, float):
        print(
            f"{key:20s}: {value:.6f}"
        )
    else:
        print(
            f"{key:20s}: {value}"
        )

# ----------------------------------------------------------------------------
# 17. IMPORTANT:
# DO NOT EVALUATE TASK 1 TEST SET HERE.
#
# The test set must remain untouched until final evaluation.
# ----------------------------------------------------------------------------

print("\nTest set evaluation: NOT PERFORMED")
print("Reason: test data must remain untouched during training/checkpoint selection.")

# ----------------------------------------------------------------------------
# 18. SAVE FINAL TRAINING CONFIGURATION
# ----------------------------------------------------------------------------

gce_config = {
    "seed": SEED,
    "task": "Task 1 — BKL vs MEL",
    "classes": {
        "bkl": 0,
        "mel": 1
    },
    "backbone": "ResNet-50",
    "pretrained": False,
    "image_size": 224,
    "batch_size": 32,
    "optimizer": "Adam",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "loss": "Generalized Cross Entropy",
    "gce_q": GCE_Q,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "checkpoint_selection": (
        "highest validation accuracy; "
        "lower absolute EOD used as tie-break"
    ),
    "normalization_source": "training split only",
    "test_used_during_training": False,
    "best_epoch": best_epoch,
    "training_time_minutes":
        training_time / 60
}

GCE_CONFIG_FILE = os.path.join(
    OUTPUT_DIR,
    "task1_gce_config.json"
)

with open(
    GCE_CONFIG_FILE,
    "w"
) as f:

    json.dump(
        gce_config,
        f,
        indent=4
    )

# ----------------------------------------------------------------------------
# 19. FINAL STEP 4 SUMMARY
# ----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 4 SUMMARY")
print("=" * 80)

print("Task             : Task 1 — BKL vs MEL")
print("Backbone         : ResNet-50")
print("Pretrained       : False")
print("GCE q            :", GCE_Q)
print("Max epochs       :", MAX_EPOCHS)
print("Best epoch       :", best_epoch)
print("Batch size       :", 32)
print("Learning rate    :", LEARNING_RATE)

print("\nBest validation results:")

print(
    "Validation loss  :",
    f"{best_val_metrics['loss']:.6f}"
)

print(
    "Validation acc   :",
    f"{best_val_metrics['accuracy']:.6f}"
)

print(
    "Validation EOD   :",
    f"{best_val_metrics['eod']:.6f}"
)

print(
    "Validation |EOD| :",
    f"{best_val_metrics['abs_eod']:.6f}"
)

print("\nFiles:")
print(
    "Best checkpoint  :",
    GCE_CHECKPOINT
)

print(
    "Training history :",
    GCE_HISTORY_FILE
)

print(
    "Configuration    :",
    GCE_CONFIG_FILE
)

print("\nTest set: NOT USED")

print("\n" + "=" * 80)
print("STEP 4 COMPLETE")
print("=" * 80)

BIASPRUNER — STEP 4 TASK 1 GCE TRAINING
Device: cuda
GPU: Tesla T4

GCE configuration:
q                    : 0.7
Maximum epochs       : 30
Early stopping       : 5
Learning rate        : 0.0001
Weight decay         : 0.0
Number of classes    : 2

Step 3 variables: PASS

Task 1 validation samples: 493

--------------------------------------------------------------------------------
CREATING RESNET-50
--------------------------------------------------------------------------------
Linear(in_features=2048, out_features=2, bias=True)

Total parameters    : 23,512,130
Trainable parameters: 23,512,130

Output directory:
/kaggle/working/BiasPruner_HAM10000

STARTING TASK 1 GCE TRAINING
Epoch 001/30 | Train Loss 0.4504 | Train Acc 0.6550 | Val Loss 0.4531 | Val Acc 0.6572 | Val EOD 0.1714 | |EOD| 0.1714 | 20.0s
Epoch 002/30 | Train Loss 0.3727 | Train Acc 0.7238 | Val Loss 0.4456 | Val Acc 0.6592 | Val EOD 0.0454 | |EOD| 0.0454 | 20.7s
Epoch 003/30 | Train Loss 0.3501 | Train Acc 0.7384 | Val

In [10]:
# ============================================================
# STEP 5 — TASK 1 BIAS ANALYSIS + PRUNING PLAN
# ============================================================
#
# Purpose:
#   1. Load the best Task-1 GCE model from Step 4
#   2. Identify EASY and HARD samples using tau = 0.7
#   3. Collect spatial activation variance for each ResNet-50
#      Bottleneck output channel
#   4. Compute BiasPruner-style bias scores
#   5. Rank units by bias score
#   6. Prune top gamma = 0.6 fraction
#   7. Save bias scores and pruning plan
#
# IMPORTANT:
#   - TRAINING DATA ONLY
#   - Validation/test data are NOT used here
#   - No model retraining happens in this step
#
# ============================================================

import os
import json
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision.models import resnet50

# ------------------------------------------------------------
# 0. Configuration
# ------------------------------------------------------------

SEED = 42

TAU = 0.7          # confidence threshold
GAMMA = 0.6        # fraction of highest-bias units to prune

CHECKPOINT_PATH = "/kaggle/working/BiasPruner_HAM10000/task1_gce_best.pt"

OUTPUT_DIR = "/kaggle/working/BiasPruner_HAM10000"

BIAS_SCORE_PATH = os.path.join(
    OUTPUT_DIR,
    "task1_bias_scores.csv"
)

PRUNING_PLAN_PATH = os.path.join(
    OUTPUT_DIR,
    "task1_pruning_plan.csv"
)

SUMMARY_PATH = os.path.join(
    OUTPUT_DIR,
    "task1_bias_analysis_summary.json"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("STEP 5 — TASK 1 BIAS ANALYSIS")
print("=" * 70)

print(f"Device: {device}")
print(f"TAU: {TAU}")
print(f"GAMMA: {GAMMA}")
print(f"Checkpoint: {CHECKPOINT_PATH}")

# ------------------------------------------------------------
# 2. Check required objects/files
# ------------------------------------------------------------

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(
        f"Task-1 GCE checkpoint not found:\n{CHECKPOINT_PATH}\n"
        "Please run Step 4 first."
    )

# We need the Task-1 training dataframe and dataloader
required_objects = [
    "task1_train_df",
    "task1_train_loader"
]

for obj_name in required_objects:
    if obj_name not in globals():
        raise RuntimeError(
            f"Required object '{obj_name}' is not available.\n"
            "Please run Step 3 first."
        )

print("\nRequired Step-3 objects found.")

# ------------------------------------------------------------
# 3. Build the same Task-1 ResNet-50 architecture
# ------------------------------------------------------------

print("\nLoading Task-1 GCE model...")

model = resnet50(weights=None)

# Task 1 = BKL vs MEL = 2 classes
model.fc = nn.Linear(
    model.fc.in_features,
    2
)

# ------------------------------------------------------------
# 4. Load best GCE checkpoint
# ------------------------------------------------------------

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device
)

# Handle either:
#   checkpoint = state_dict
# OR
#   checkpoint["model_state_dict"]
# OR
#   checkpoint["state_dict"]

if isinstance(checkpoint, dict):

    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]

    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]

    else:
        # Assume the dictionary itself is the state_dict
        state_dict = checkpoint

else:
    raise RuntimeError(
        "Unexpected checkpoint format."
    )

# Remove possible "module." prefix
clean_state_dict = {}

for key, value in state_dict.items():

    if key.startswith("module."):
        key = key[len("module."):]

    clean_state_dict[key] = value

model.load_state_dict(
    clean_state_dict,
    strict=True
)

model = model.to(device)
model.eval()

print("Task-1 GCE checkpoint loaded successfully.")

# ------------------------------------------------------------
# 5. Identify ResNet-50 Bottleneck blocks
# ------------------------------------------------------------

# We use the output of every Bottleneck block.
#
# Each Bottleneck output has C channels.
# Every channel is treated as one "unit".
#
# ResNet-50:
#
# layer1:
#   3 blocks × 256 channels = 768
#
# layer2:
#   4 blocks × 512 channels = 2048
#
# layer3:
#   6 blocks × 1024 channels = 6144
#
# layer4:
#   3 blocks × 2048 channels = 6144
#
# TOTAL:
#   768 + 2048 + 6144 + 6144
#   = 15104 units
#
# This 15104 value is calculated automatically below.
# ------------------------------------------------------------

bottleneck_blocks = []

for layer_name in ["layer1", "layer2", "layer3", "layer4"]:

    layer = getattr(model, layer_name)

    for block_idx, block in enumerate(layer):

        bottleneck_blocks.append(
            {
                "layer": layer_name,
                "block_index": block_idx,
                "module": block
            }
        )

print("\nResNet-50 Bottleneck blocks:")

for item in bottleneck_blocks:

    block = item["module"]

    print(
        f"{item['layer']} "
        f"block {item['block_index']} "
        f"-> output channels = {block.conv3.out_channels}"
    )

# ------------------------------------------------------------
# 6. Calculate total number of units
# ------------------------------------------------------------

unit_counts = []

for item in bottleneck_blocks:

    block = item["module"]

    unit_counts.append(
        block.conv3.out_channels
    )

TOTAL_UNITS = sum(unit_counts)

print("\n" + "-" * 70)
print(f"Total scored units: {TOTAL_UNITS}")

# Correct ResNet-50 value
EXPECTED_UNITS = 15104

if TOTAL_UNITS != EXPECTED_UNITS:

    raise RuntimeError(
        f"Unexpected total units.\n"
        f"Expected: {EXPECTED_UNITS}\n"
        f"Found: {TOTAL_UNITS}"
    )

print("Unit-count sanity check: PASS")
print("-" * 70)

# ------------------------------------------------------------
# 7. Create activation hooks
# ------------------------------------------------------------

activations = {}

hooks = []


def make_hook(layer_name, block_index):

    key = f"{layer_name}.block{block_index}"

    def hook(module, inputs, output):

        # output shape:
        #
        # [batch, channels, height, width]
        #
        activations[key] = output.detach()

    return hook


for item in bottleneck_blocks:

    layer_name = item["layer"]
    block_index = item["block_index"]
    block = item["module"]

    hook_handle = block.register_forward_hook(
        make_hook(
            layer_name,
            block_index
        )
    )

    hooks.append(hook_handle)

print(f"\nRegistered {len(hooks)} activation hooks.")

# ------------------------------------------------------------
# 8. Prepare Task-1 training dataset
# ------------------------------------------------------------

# IMPORTANT:
# We explicitly use Task-1 TRAINING data.
#
# No validation.
# No test.

task1_train_dataset = task1_train_loader.dataset

print("\nTask-1 training samples:")
print(len(task1_train_dataset))

# ------------------------------------------------------------
# 9. Create accumulators
# ------------------------------------------------------------

# For each class:
#
# Easy:
#   true class = c
#   prediction = c
#   confidence >= tau
#
# Hard:
#   true class != c
#   prediction = c
#   confidence >= tau
#
# For every unit:
#
#   mean variance on EASY samples
#   mean variance on HARD samples
#
# Bias score:
#
# S(c,n) =
#       mean_easy_variance
#       -
#       mean_hard_variance
#
# Then:
#
# S_bar(n) = average over classes
#
# ------------------------------------------------------------

easy_count = {
    0: 0,
    1: 0
}

hard_count = {
    0: 0,
    1: 0
}

# Each key will contain:
#
#   easy_sum
#   hard_sum
#
# for every unit.

easy_variance_sum = {}
hard_variance_sum = {}

for item in bottleneck_blocks:

    layer_name = item["layer"]
    block_index = item["block_index"]

    key = f"{layer_name}.block{block_index}"

    channels = item["module"].conv3.out_channels

    easy_variance_sum[key] = np.zeros(
        channels,
        dtype=np.float64
    )

    hard_variance_sum[key] = np.zeros(
        channels,
        dtype=np.float64
    )

# ------------------------------------------------------------
# 10. Analyze training data
# ------------------------------------------------------------

print("\nStarting bias analysis...")
print("Only Task-1 training data will be used.")

with torch.no_grad():

    for batch_idx, (images, labels) in enumerate(
        task1_train_loader
    ):

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        # ----------------------------------------------------
        # Forward pass
        # ----------------------------------------------------

        logits = model(images)

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        confidence, predictions = torch.max(
            probabilities,
            dim=1
        )

        # ----------------------------------------------------
        # Determine EASY/HARD masks
        # ----------------------------------------------------

        for class_id in [0, 1]:

            # EASY samples for class c:
            #
            # y = c
            # prediction = c
            # confidence >= tau

            easy_mask = (
                (labels == class_id)
                &
                (predictions == class_id)
                &
                (confidence >= TAU)
            )

            # HARD samples for class c:
            #
            # y != c
            # prediction = c
            # confidence >= tau

            hard_mask = (
                (labels != class_id)
                &
                (predictions == class_id)
                &
                (confidence >= TAU)
            )

            easy_indices = torch.where(
                easy_mask
            )[0]

            hard_indices = torch.where(
                hard_mask
            )[0]

            easy_count[class_id] += len(
                easy_indices
            )

            hard_count[class_id] += len(
                hard_indices
            )

            # ------------------------------------------------
            # Calculate spatial variance
            # ------------------------------------------------

            if len(easy_indices) > 0:

                for item in bottleneck_blocks:

                    layer_name = item["layer"]
                    block_index = item["block_index"]

                    key = (
                        f"{layer_name}.block{block_index}"
                    )

                    activation = activations[key]

                    # Select easy samples
                    easy_activation = activation[
                        easy_indices
                    ]

                    # Shape:
                    #
                    # [N, C, H, W]
                    #
                    # Variance across H,W
                    #

                    variance = torch.var(
                        easy_activation,
                        dim=(2, 3),
                        unbiased=False
                    )

                    # Average over samples
                    variance = variance.mean(
                        dim=0
                    )

                    easy_variance_sum[key] += (
                        variance.cpu().numpy()
                    )

            if len(hard_indices) > 0:

                for item in bottleneck_blocks:

                    layer_name = item["layer"]
                    block_index = item["block_index"]

                    key = (
                        f"{layer_name}.block{block_index}"
                    )

                    activation = activations[key]

                    # Select hard samples
                    hard_activation = activation[
                        hard_indices
                    ]

                    # Spatial variance
                    variance = torch.var(
                        hard_activation,
                        dim=(2, 3),
                        unbiased=False
                    )

                    # Average over samples
                    variance = variance.mean(
                        dim=0
                    )

                    hard_variance_sum[key] += (
                        variance.cpu().numpy()
                    )

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if (
            (batch_idx + 1) % 10 == 0
            or
            (batch_idx + 1) == len(task1_train_loader)
        ):

            print(
                f"Processed batch "
                f"{batch_idx + 1}/"
                f"{len(task1_train_loader)}"
            )

# ------------------------------------------------------------
# 11. Remove hooks
# ------------------------------------------------------------

for hook in hooks:
    hook.remove()

print("\nActivation hooks removed.")

# ------------------------------------------------------------
# 12. Print Easy/Hard counts
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EASY / HARD SAMPLE COUNTS")
print("=" * 70)

for class_id in [0, 1]:

    print(
        f"Class {class_id}: "
        f"Easy = {easy_count[class_id]}, "
        f"Hard = {hard_count[class_id]}"
    )

# ------------------------------------------------------------
# 13. Calculate bias scores
# ------------------------------------------------------------

print("\nCalculating unit-level bias scores...")

rows = []

global_unit_id = 0

for item in bottleneck_blocks:

    layer_name = item["layer"]
    block_index = item["block_index"]

    key = (
        f"{layer_name}.block{block_index}"
    )

    channels = item["module"].conv3.out_channels

    for channel_id in range(channels):

        class_scores = []

        # -----------------------------------------------
        # Class 0
        # -----------------------------------------------

        if easy_count[0] > 0:
            easy_mean_0 = (
                easy_variance_sum[key][channel_id]
                / easy_count[0]
            )
        else:
            easy_mean_0 = np.nan

        if hard_count[0] > 0:
            hard_mean_0 = (
                hard_variance_sum[key][channel_id]
                / hard_count[0]
            )
        else:
            hard_mean_0 = np.nan

        score_0 = (
            easy_mean_0 - hard_mean_0
            if not np.isnan(easy_mean_0)
            and not np.isnan(hard_mean_0)
            else np.nan
        )

        # -----------------------------------------------
        # Class 1
        # -----------------------------------------------

        if easy_count[1] > 0:
            easy_mean_1 = (
                easy_variance_sum[key][channel_id]
                / easy_count[1]
            )
        else:
            easy_mean_1 = np.nan

        if hard_count[1] > 0:
            hard_mean_1 = (
                hard_variance_sum[key][channel_id]
                / hard_count[1]
            )
        else:
            hard_mean_1 = np.nan

        score_1 = (
            easy_mean_1 - hard_mean_1
            if not np.isnan(easy_mean_1)
            and not np.isnan(hard_mean_1)
            else np.nan
        )

        # -----------------------------------------------
        # Aggregate score across classes
        # -----------------------------------------------

        valid_scores = [
            score
            for score in [score_0, score_1]
            if not np.isnan(score)
        ]

        if len(valid_scores) > 0:

            final_score = float(
                np.mean(valid_scores)
            )

        else:

            final_score = np.nan

        rows.append(
            {
                "global_unit_id": global_unit_id,
                "layer": layer_name,
                "block_index": block_index,
                "channel": channel_id,

                "class0_bias_score": score_0,
                "class1_bias_score": score_1,

                "bias_score": final_score
            }
        )

        global_unit_id += 1

# ------------------------------------------------------------
# 14. Convert to dataframe
# ------------------------------------------------------------

bias_df = pd.DataFrame(rows)

print(
    f"\nBias-score dataframe shape: "
    f"{bias_df.shape}"
)

if len(bias_df) != TOTAL_UNITS:

    raise RuntimeError(
        f"Bias dataframe contains "
        f"{len(bias_df)} units, "
        f"expected {TOTAL_UNITS}."
    )

print("Bias-score unit-count check: PASS")

# ------------------------------------------------------------
# 15. Handle NaN scores
# ------------------------------------------------------------

valid_bias_df = bias_df[
    bias_df["bias_score"].notna()
].copy()

invalid_count = (
    bias_df["bias_score"].isna().sum()
)

print(
    f"Valid bias scores: {len(valid_bias_df)}"
)

print(
    f"NaN bias scores: {invalid_count}"
)

if len(valid_bias_df) == 0:

    raise RuntimeError(
        "No valid bias scores were produced."
    )

# ------------------------------------------------------------
# 16. Rank units
# ------------------------------------------------------------

# Higher bias score = more biased toward easy
# vs hard behavior.
#
# Therefore highest scores are selected for pruning.

valid_bias_df = valid_bias_df.sort_values(
    by="bias_score",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# 17. Determine pruning count
# ------------------------------------------------------------

NUM_TO_PRUNE = math.ceil(
    GAMMA * TOTAL_UNITS
)

NUM_TO_RETAIN = (
    TOTAL_UNITS - NUM_TO_PRUNE
)

print("\n" + "=" * 70)
print("PRUNING PLAN")
print("=" * 70)

print(
    f"Total units: {TOTAL_UNITS}"
)

print(
    f"Gamma: {GAMMA}"
)

print(
    f"Units to prune: {NUM_TO_PRUNE}"
)

print(
    f"Units to retain: {NUM_TO_RETAIN}"
)

# ------------------------------------------------------------
# 18. Create pruning mask
# ------------------------------------------------------------

# Start with everything retained.
#
# prune = 1
# retain = 0

bias_df["prune"] = 0

top_units = valid_bias_df.head(
    NUM_TO_PRUNE
)["global_unit_id"].values

bias_df.loc[
    bias_df["global_unit_id"].isin(top_units),
    "prune"
] = 1

# ------------------------------------------------------------
# 19. Add rank
# ------------------------------------------------------------

rank_map = {
    unit_id: rank + 1
    for rank, unit_id
    in enumerate(
        valid_bias_df["global_unit_id"].values
    )
}

bias_df["bias_rank"] = (
    bias_df["global_unit_id"]
    .map(rank_map)
)

# ------------------------------------------------------------
# 20. Add retain/prune label
# ------------------------------------------------------------

bias_df["status"] = np.where(
    bias_df["prune"] == 1,
    "PRUNE",
    "RETAIN"
)

# ------------------------------------------------------------
# 21. Save complete bias scores
# ------------------------------------------------------------

bias_df.to_csv(
    BIAS_SCORE_PATH,
    index=False
)

print(
    f"\nSaved bias scores:\n"
    f"{BIAS_SCORE_PATH}"
)

# ------------------------------------------------------------
# 22. Save pruning plan
# ------------------------------------------------------------

pruning_plan = bias_df[
    [
        "global_unit_id",
        "layer",
        "block_index",
        "channel",
        "bias_score",
        "bias_rank",
        "prune",
        "status"
    ]
].copy()

pruning_plan.to_csv(
    PRUNING_PLAN_PATH,
    index=False
)

print(
    f"Saved pruning plan:\n"
    f"{PRUNING_PLAN_PATH}"
)

# ------------------------------------------------------------
# 23. Create summary statistics
# ------------------------------------------------------------

summary = {

    "step": 5,

    "dataset": "HAM10000",

    "task": "BKL_vs_MEL",

    "seed": SEED,

    "tau": TAU,

    "gamma": GAMMA,

    "total_units": int(TOTAL_UNITS),

    "units_pruned": int(NUM_TO_PRUNE),

    "units_retained": int(NUM_TO_RETAIN),

    "easy_count_class_0": int(
        easy_count[0]
    ),

    "hard_count_class_0": int(
        hard_count[0]
    ),

    "easy_count_class_1": int(
        easy_count[1]
    ),

    "hard_count_class_1": int(
        hard_count[1]
    ),

    "valid_bias_scores": int(
        len(valid_bias_df)
    ),

    "nan_bias_scores": int(
        invalid_count
    ),

    "bias_score_min": float(
        valid_bias_df["bias_score"].min()
    ),

    "bias_score_max": float(
        valid_bias_df["bias_score"].max()
    ),

    "bias_score_mean": float(
        valid_bias_df["bias_score"].mean()
    ),

    "bias_score_median": float(
        valid_bias_df["bias_score"].median()
    ),

    "checkpoint": CHECKPOINT_PATH,

    "data_used": "Task-1 training split only",

    "validation_used": False,

    "test_used": False,

    "activation_definition": (
        "Output of each ResNet-50 Bottleneck block; "
        "spatial variance computed over H x W"
    ),

    "implementation_note": (
        "The paper defines unit-level bias scores using "
        "feature-map spatial variance but does not fully "
        "specify the exact torchvision ResNet-50 tensor "
        "location. This implementation uses the output "
        "of each Bottleneck block as the unit representation."
    )
}

with open(
    SUMMARY_PATH,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

print(
    f"Saved summary:\n"
    f"{SUMMARY_PATH}"
)

# ------------------------------------------------------------
# 24. Print top/bottom units
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TOP 10 HIGHEST-BIAS UNITS")
print("=" * 70)

print(
    valid_bias_df[
        [
            "global_unit_id",
            "layer",
            "block_index",
            "channel",
            "bias_score"
        ]
    ].head(10).to_string(index=False)
)

print("\n" + "=" * 70)
print("BOTTOM 10 BIAS-SCORE UNITS")
print("=" * 70)

print(
    valid_bias_df[
        [
            "global_unit_id",
            "layer",
            "block_index",
            "channel",
            "bias_score"
        ]
    ].tail(10).to_string(index=False)
)

# ------------------------------------------------------------
# 25. Final sanity checks
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 5 SANITY CHECKS")
print("=" * 70)

# Check 1
assert len(bias_df) == 15104
print("1. Total units = 15104: PASS")

# Check 2
assert (
    bias_df["prune"].sum()
    == NUM_TO_PRUNE
)
print(
    f"2. Pruned units = {NUM_TO_PRUNE}: PASS"
)

# Check 3
assert (
    (bias_df["prune"] == 0).sum()
    == NUM_TO_RETAIN
)
print(
    f"3. Retained units = {NUM_TO_RETAIN}: PASS"
)

# Check 4
assert (
    easy_count[0] > 0
    and easy_count[1] > 0
)
print("4. Easy samples available for both classes: PASS")

# Check 5
assert (
    hard_count[0] > 0
    and hard_count[1] > 0
)
print("5. Hard samples available for both classes: PASS")

# Check 6
assert (
    bias_df["global_unit_id"].nunique()
    == TOTAL_UNITS
)
print("6. Unique unit IDs: PASS")

print("\n" + "=" * 70)
print("STEP 5 COMPLETE — PASS")
print("=" * 70)

print("\nFiles created:")

print(
    f"1. {BIAS_SCORE_PATH}"
)

print(
    f"2. {PRUNING_PLAN_PATH}"
)

print(
    f"3. {SUMMARY_PATH}"
)

print("\nDo NOT start Step 6 until we inspect this output.")

STEP 5 — TASK 1 BIAS ANALYSIS
Device: cuda
TAU: 0.7
GAMMA: 0.6
Checkpoint: /kaggle/working/BiasPruner_HAM10000/task1_gce_best.pt

Required Step-3 objects found.

Loading Task-1 GCE model...
Task-1 GCE checkpoint loaded successfully.

ResNet-50 Bottleneck blocks:
layer1 block 0 -> output channels = 256
layer1 block 1 -> output channels = 256
layer1 block 2 -> output channels = 256
layer2 block 0 -> output channels = 512
layer2 block 1 -> output channels = 512
layer2 block 2 -> output channels = 512
layer2 block 3 -> output channels = 512
layer3 block 0 -> output channels = 1024
layer3 block 1 -> output channels = 1024
layer3 block 2 -> output channels = 1024
layer3 block 3 -> output channels = 1024
layer3 block 4 -> output channels = 1024
layer3 block 5 -> output channels = 1024
layer4 block 0 -> output channels = 2048
layer4 block 1 -> output channels = 2048
layer4 block 2 -> output channels = 2048

----------------------------------------------------------------------
Total scored uni

In [11]:
# ============================================================
# STEP 6 — TASK 1 PRUNING + WCE FINE-TUNING
# ============================================================
#
# Task 1:
#   BKL -> 0
#   MEL -> 1
#
# Starting model:
#   Best Task-1 GCE checkpoint from Step 4
#
# Pruning:
#   Bias scores from Step 5
#   gamma = 0.6
#   15,104 total units
#   9,063 pruned
#   6,041 retained
#
# Fine-tuning:
#   Weighted Cross Entropy (WCE)
#   W(x) = exp(alpha * L_GCE(x))
#   alpha is TRAINABLE and constrained to 0 < alpha < 1
#   20 epochs
#
# Validation:
#   Used for checkpoint selection only
#
# Test:
#   Used ONLY for final evaluation
#
# ============================================================

import os
import json
import math
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision.models import resnet50


# ============================================================
# 0. CONFIGURATION
# ============================================================

SEED = 42

NUM_CLASSES = 2

Q = 0.7

GAMMA = 0.6

WCE_EPOCHS = 20

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 0.0

# Initial alpha
INITIAL_ALPHA = 0.5

CHECKPOINT_GCE = (
    "/kaggle/working/BiasPruner_HAM10000/"
    "task1_gce_best.pt"
)

BIAS_SCORE_PATH = (
    "/kaggle/working/BiasPruner_HAM10000/"
    "task1_bias_scores.csv"
)

PRUNING_PLAN_PATH = (
    "/kaggle/working/BiasPruner_HAM10000/"
    "task1_pruning_plan.csv"
)

OUTPUT_DIR = (
    "/kaggle/working/BiasPruner_HAM10000"
)

FINAL_CHECKPOINT = os.path.join(
    OUTPUT_DIR,
    "task1_final.pt"
)

HISTORY_PATH = os.path.join(
    OUTPUT_DIR,
    "task1_wce_history.csv"
)

CONFIG_PATH = os.path.join(
    OUTPUT_DIR,
    "task1_wce_config.json"
)

TEST_RESULTS_PATH = os.path.join(
    OUTPUT_DIR,
    "task1_test_results.json"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("STEP 6 — TASK 1 PRUNING + WCE FINE-TUNING")
print("=" * 70)

print(f"Device: {device}")
print(f"Q: {Q}")
print(f"Gamma: {GAMMA}")
print(f"WCE epochs: {WCE_EPOCHS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Initial alpha: {INITIAL_ALPHA}")


# ============================================================
# 2. CHECK REQUIRED FILES / OBJECTS
# ============================================================

for path in [
    CHECKPOINT_GCE,
    BIAS_SCORE_PATH,
    PRUNING_PLAN_PATH
]:

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"Required file not found:\n{path}\n"
            "Please complete Steps 4 and 5 first."
        )


required_objects = [
    "task1_train_loader",
    "task1_val_loader",
    "task1_test_loader"
]

for obj_name in required_objects:

    if obj_name not in globals():

        raise RuntimeError(
            f"Required object '{obj_name}' "
            "is not available.\n"
            "Please run Step 3 first."
        )

print("\nRequired files and dataloaders found.")


# ============================================================
# 3. LOAD PRUNING PLAN
# ============================================================

print("\nLoading Step-5 pruning plan...")

pruning_plan = pd.read_csv(
    PRUNING_PLAN_PATH
)

bias_scores = pd.read_csv(
    BIAS_SCORE_PATH
)

print(
    f"Pruning plan rows: {len(pruning_plan)}"
)

print(
    f"Bias-score rows: {len(bias_scores)}"
)


# ------------------------------------------------------------
# Validate total units
# ------------------------------------------------------------

EXPECTED_UNITS = 15104

if len(pruning_plan) != EXPECTED_UNITS:

    raise RuntimeError(
        f"Expected {EXPECTED_UNITS} pruning-plan rows, "
        f"found {len(pruning_plan)}."
    )


# ------------------------------------------------------------
# Validate pruning count
# ------------------------------------------------------------

num_pruned = int(
    pruning_plan["prune"].sum()
)

num_retained = int(
    (pruning_plan["prune"] == 0).sum()
)

expected_pruned = math.ceil(
    GAMMA * EXPECTED_UNITS
)

expected_retained = (
    EXPECTED_UNITS - expected_pruned
)

print("\nPruning statistics:")

print(
    f"Total units: {EXPECTED_UNITS}"
)

print(
    f"Units marked PRUNE: {num_pruned}"
)

print(
    f"Units marked RETAIN: {num_retained}"
)

if num_pruned != expected_pruned:

    raise RuntimeError(
        f"Expected {expected_pruned} pruned units, "
        f"found {num_pruned}."
    )

if num_retained != expected_retained:

    raise RuntimeError(
        f"Expected {expected_retained} retained units, "
        f"found {num_retained}."
    )

print("Pruning-plan validation: PASS")


# ============================================================
# 4. BUILD RESNET-50
# ============================================================

print("\nBuilding ResNet-50...")

model = resnet50(
    weights=None
)

model.fc = nn.Linear(
    model.fc.in_features,
    NUM_CLASSES
)


# ============================================================
# 5. LOAD BEST GCE CHECKPOINT
# ============================================================

print("Loading Task-1 GCE checkpoint...")

checkpoint = torch.load(
    CHECKPOINT_GCE,
    map_location=device
)

if isinstance(checkpoint, dict):

    if "model_state_dict" in checkpoint:

        state_dict = checkpoint[
            "model_state_dict"
        ]

    elif "state_dict" in checkpoint:

        state_dict = checkpoint[
            "state_dict"
        ]

    else:

        state_dict = checkpoint

else:

    raise RuntimeError(
        "Unexpected GCE checkpoint format."
    )


# Remove possible DataParallel prefix

clean_state_dict = {}

for key, value in state_dict.items():

    if key.startswith("module."):

        key = key[len("module."):]

    clean_state_dict[key] = value


model.load_state_dict(
    clean_state_dict,
    strict=True
)

model = model.to(device)

print(
    "GCE checkpoint loaded successfully."
)


# ============================================================
# 6. CREATE PRUNING MASKS
# ============================================================
#
# One mask per ResNet-50 Bottleneck output.
#
# mask[channel] = 1
#     retained
#
# mask[channel] = 0
#     pruned
#
# The ordering must exactly match Step 5:
#
# layer1 block0
# layer1 block1
# layer1 block2
# layer2 block0
# ...
# layer4 block2
#
# ============================================================

pruning_masks = {}

global_unit_id = 0

for layer_name in [
    "layer1",
    "layer2",
    "layer3",
    "layer4"
]:

    layer = getattr(
        model,
        layer_name
    )

    for block_index, block in enumerate(layer):

        key = (
            f"{layer_name}.block{block_index}"
        )

        channels = (
            block.conv3.out_channels
        )

        mask = torch.ones(
            channels,
            dtype=torch.float32
        )

        for channel in range(channels):

            row = pruning_plan[
                pruning_plan["global_unit_id"]
                == global_unit_id
            ]

            if len(row) != 1:

                raise RuntimeError(
                    f"Could not uniquely find "
                    f"unit {global_unit_id}."
                )

            prune_value = int(
                row.iloc[0]["prune"]
            )

            if prune_value == 1:

                mask[channel] = 0.0

            global_unit_id += 1

        pruning_masks[key] = mask.to(
            device
        )


if global_unit_id != EXPECTED_UNITS:

    raise RuntimeError(
        f"Created {global_unit_id} units, "
        f"expected {EXPECTED_UNITS}."
    )

print(
    f"\nCreated masks for {len(pruning_masks)} "
    "Bottleneck blocks."
)


# ============================================================
# 7. REGISTER PRUNING HOOKS
# ============================================================
#
# IMPORTANT:
#
# The same Bottleneck output representation was used in Step 5
# for bias scoring.
#
# Here we apply the pruning mask to that same representation.
#
# ============================================================

pruning_hooks = []


def make_pruning_hook(mask):

    def hook(module, inputs, output):

        # output:
        # [B, C, H, W]
        #
        # mask:
        # [C]
        #
        # reshape mask to:
        # [1, C, 1, 1]

        return (
            output
            * mask.view(
                1,
                -1,
                1,
                1
            )
        )

    return hook


for layer_name in [
    "layer1",
    "layer2",
    "layer3",
    "layer4"
]:

    layer = getattr(
        model,
        layer_name
    )

    for block_index, block in enumerate(layer):

        key = (
            f"{layer_name}.block{block_index}"
        )

        handle = block.register_forward_hook(
            make_pruning_hook(
                pruning_masks[key]
            )
        )

        pruning_hooks.append(handle)


print(
    f"Registered {len(pruning_hooks)} pruning hooks."
)


# ============================================================
# 8. VERIFY PRUNING MASKS
# ============================================================

total_mask_values = 0
total_zero_values = 0
total_one_values = 0

for key, mask in pruning_masks.items():

    total_mask_values += mask.numel()

    total_zero_values += int(
        (mask == 0).sum().item()
    )

    total_one_values += int(
        (mask == 1).sum().item()
    )


print("\nMask verification:")

print(
    f"Mask values: {total_mask_values}"
)

print(
    f"Zero/pruned: {total_zero_values}"
)

print(
    f"One/retained: {total_one_values}"
)

assert total_mask_values == EXPECTED_UNITS

assert total_zero_values == expected_pruned

assert total_one_values == expected_retained

print("Mask verification: PASS")


# ============================================================
# 9. TRAINABLE ALPHA
# ============================================================
#
# We need:
#
#       0 < alpha < 1
#
# Instead of directly optimizing alpha, optimize an unconstrained
# parameter alpha_raw and transform it through sigmoid:
#
#       alpha = sigmoid(alpha_raw)
#
# Therefore alpha is always between 0 and 1.
#
# ============================================================

initial_alpha = float(
    INITIAL_ALPHA
)

if not (
    0.0 < initial_alpha < 1.0
):

    raise ValueError(
        "INITIAL_ALPHA must be between 0 and 1."
    )


initial_alpha_raw = math.log(
    initial_alpha
    /
    (1.0 - initial_alpha)
)


alpha_raw = nn.Parameter(
    torch.tensor(
        initial_alpha_raw,
        dtype=torch.float32,
        device=device
    )
)


def get_alpha():

    return torch.sigmoid(
        alpha_raw
    )


print(
    f"\nInitial trainable alpha: "
    f"{get_alpha().item():.6f}"
)


# ============================================================
# 10. GENERALIZED CROSS ENTROPY
# ============================================================

def generalized_cross_entropy(
    logits,
    targets,
    q=0.7
):

    probabilities = torch.softmax(
        logits,
        dim=1
    )

    true_probability = probabilities[
        torch.arange(
            targets.size(0),
            device=targets.device
        ),
        targets
    ]

    true_probability = torch.clamp(
        true_probability,
        min=1e-8,
        max=1.0
    )

    gce = (
        1.0
        -
        true_probability.pow(q)
    ) / q

    return gce


# ============================================================
# 11. WCE LOSS
# ============================================================
#
# Paper formulation:
#
# L_WCE(x)
#     =
# W(x) * L_CE(x,y)
#
# where
#
# W(x)
#     =
# exp(alpha * L_GCE(x))
#
# alpha is trainable.
#
# ============================================================

def weighted_cross_entropy(
    logits,
    targets,
    alpha,
    q=0.7
):

    # Per-sample CE
    ce_loss = F.cross_entropy(
        logits,
        targets,
        reduction="none"
    )

    # Per-sample GCE
    gce_loss = generalized_cross_entropy(
        logits,
        targets,
        q=q
    )

    # Weight
    weights = torch.exp(
        alpha * gce_loss
    )

    weighted_loss = (
        weights * ce_loss
    )

    return weighted_loss.mean()


# ============================================================
# 12. OPTIMIZER
# ============================================================
#
# Include:
#   model parameters
#   alpha_raw
#
# Pruned activations are masked during forward propagation.
#
# ============================================================

optimizer = torch.optim.Adam(
    list(model.parameters())
    + [alpha_raw],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# 13. METRIC FUNCTION
# ============================================================

def calculate_metrics(
    y_true,
    y_pred,
    ages
):

    y_true = np.asarray(
        y_true
    )

    y_pred = np.asarray(
        y_pred
    )

    ages = np.asarray(
        ages
    )

    accuracy = float(
        np.mean(
            y_true == y_pred
        )
    )

    # --------------------------------------------------------
    # Balanced accuracy
    # --------------------------------------------------------

    recalls = []

    for cls in [0, 1]:

        mask = (
            y_true == cls
        )

        if mask.sum() > 0:

            recalls.append(
                np.mean(
                    y_pred[mask] == cls
                )
            )

    balanced_accuracy = float(
        np.mean(recalls)
    )

    # --------------------------------------------------------
    # Macro F1
    # --------------------------------------------------------

    f1_values = []

    for cls in [0, 1]:

        tp = np.sum(
            (y_true == cls)
            &
            (y_pred == cls)
        )

        fp = np.sum(
            (y_true != cls)
            &
            (y_pred == cls)
        )

        fn = np.sum(
            (y_true == cls)
            &
            (y_pred != cls)
        )

        precision = (
            tp / (tp + fp)
            if (tp + fp) > 0
            else 0.0
        )

        recall = (
            tp / (tp + fn)
            if (tp + fn) > 0
            else 0.0
        )

        if (
            precision + recall
            > 0
        ):

            f1 = (
                2
                * precision
                * recall
                /
                (precision + recall)
            )

        else:

            f1 = 0.0

        f1_values.append(f1)

    macro_f1 = float(
        np.mean(f1_values)
    )

    # --------------------------------------------------------
    # Age-group accuracy
    # --------------------------------------------------------

    valid_age = ~np.isnan(
        ages
    )

    lt60 = (
        valid_age
        &
        (ages < 60)
    )

    ge60 = (
        valid_age
        &
        (ages >= 60)
    )

    accuracy_lt60 = (
        float(
            np.mean(
                y_true[lt60]
                ==
                y_pred[lt60]
            )
        )
        if lt60.sum() > 0
        else np.nan
    )

    accuracy_ge60 = (
        float(
            np.mean(
                y_true[ge60]
                ==
                y_pred[ge60]
            )
        )
        if ge60.sum() > 0
        else np.nan
    )

    # --------------------------------------------------------
    # TPR by age group
    #
    # Positive class = 1
    # --------------------------------------------------------

    def tpr_for_mask(mask):

        positives = (
            mask
            &
            (y_true == 1)
        )

        denominator = positives.sum()

        if denominator == 0:

            return np.nan

        return float(
            np.sum(
                y_pred[positives] == 1
            )
            /
            denominator
        )

    tpr_lt60 = tpr_for_mask(
        lt60
    )

    tpr_ge60 = tpr_for_mask(
        ge60
    )

    # --------------------------------------------------------
    # Equal Opportunity Difference
    # --------------------------------------------------------

    if (
        not np.isnan(tpr_lt60)
        and
        not np.isnan(tpr_ge60)
    ):

        eod = (
            tpr_lt60
            -
            tpr_ge60
        )

        abs_eod = abs(eod)

    else:

        eod = np.nan
        abs_eod = np.nan

    # --------------------------------------------------------
    # Demographic Parity Ratio
    #
    # P(predicted positive | group)
    # --------------------------------------------------------

    def positive_rate(mask):

        count = mask.sum()

        if count == 0:

            return np.nan

        return float(
            np.mean(
                y_pred[mask] == 1
            )
        )

    positive_rate_lt60 = positive_rate(
        lt60
    )

    positive_rate_ge60 = positive_rate(
        ge60
    )

    if (
        not np.isnan(
            positive_rate_lt60
        )
        and
        not np.isnan(
            positive_rate_ge60
        )
        and
        positive_rate_ge60 > 0
    ):

        dpr = (
            positive_rate_lt60
            /
            positive_rate_ge60
        )

    else:

        dpr = np.nan

    return {

        "accuracy": accuracy,

        "balanced_accuracy":
            balanced_accuracy,

        "macro_f1":
            macro_f1,

        "accuracy_lt60":
            accuracy_lt60,

        "accuracy_ge60":
            accuracy_ge60,

        "tpr_lt60":
            tpr_lt60,

        "tpr_ge60":
            tpr_ge60,

        "EOD":
            eod,

        "abs_EOD":
            abs_eod,

        "DPR":
            dpr,

        "positive_rate_lt60":
            positive_rate_lt60,

        "positive_rate_ge60":
            positive_rate_ge60
    }


# ============================================================
# 14. EVALUATION FUNCTION
# ============================================================

def evaluate_loader(
    model,
    loader,
    dataframe,
    device
):

    model.eval()

    all_targets = []
    all_predictions = []
    all_ages = []

    total_loss = 0.0
    total_samples = 0

    # --------------------------------------------------------
    # IMPORTANT:
    #
    # The dataframe must correspond to the same ordering as
    # the dataset underlying the loader.
    #
    # Step 3 constructed these datasets using reset_index.
    # --------------------------------------------------------

    age_values = dataframe[
        "age"
    ].to_numpy(
        dtype=float
    )

    dataset_length = len(
        loader.dataset
    )

    if len(age_values) != dataset_length:

        raise RuntimeError(
            "Dataframe length does not match "
            "loader dataset length."
        )

    offset = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            logits = model(
                images
            )

            loss = F.cross_entropy(
                logits,
                labels,
                reduction="sum"
            )

            predictions = torch.argmax(
                logits,
                dim=1
            )

            batch_size = (
                labels.size(0)
            )

            total_loss += (
                loss.item()
            )

            total_samples += (
                batch_size
            )

            all_targets.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_ages.extend(
                age_values[
                    offset:
                    offset + batch_size
                ]
            )

            offset += batch_size

    metrics = calculate_metrics(
        all_targets,
        all_predictions,
        all_ages
    )

    metrics["loss"] = (
        total_loss
        /
        total_samples
    )

    return metrics


# ============================================================
# 15. TRAINING FUNCTION
# ============================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    alpha_raw,
    device
):

    model.train()

    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(
            images
        )

        alpha = torch.sigmoid(
            alpha_raw
        )

        loss = weighted_cross_entropy(
            logits,
            labels,
            alpha,
            q=Q
        )

        loss.backward()

        optimizer.step()

        batch_size = (
            labels.size(0)
        )

        running_loss += (
            loss.item()
            *
            batch_size
        )

        predictions = torch.argmax(
            logits,
            dim=1
        )

        running_correct += int(
            (
                predictions == labels
            ).sum().item()
        )

        total += batch_size

    return {

        "loss":
            running_loss / total,

        "accuracy":
            running_correct / total,

        "alpha":
            float(
                torch.sigmoid(
                    alpha_raw
                ).item()
            )
    }


# ============================================================
# 16. INITIAL PRUNED-MODEL CHECK
# ============================================================

print("\nRunning initial pruned-model forward pass...")

model.eval()

with torch.no_grad():

    sample_images, sample_labels = next(
        iter(task1_train_loader)
    )

    sample_images = sample_images.to(
        device
    )

    sample_logits = model(
        sample_images
    )

    sample_predictions = torch.argmax(
        sample_logits,
        dim=1
    )

print(
    f"Sample logits shape: "
    f"{tuple(sample_logits.shape)}"
)

print(
    f"Sample prediction shape: "
    f"{tuple(sample_predictions.shape)}"
)

assert sample_logits.shape[1] == 2

assert torch.isfinite(
    sample_logits
).all()

print(
    "Initial pruned-model forward check: PASS"
)


# ============================================================
# 17. WCE FINE-TUNING
# ============================================================

print("\n" + "=" * 70)
print("STARTING WCE FINE-TUNING")
print("=" * 70)

history = []

best_epoch = None
best_val_accuracy = -float("inf")
best_val_abs_eod = float("inf")

best_state_dict = None
best_alpha_raw = None


for epoch in range(
    1,
    WCE_EPOCHS + 1
):

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    train_metrics = train_one_epoch(
        model,
        task1_train_loader,
        optimizer,
        alpha_raw,
        device
    )

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    val_metrics = evaluate_loader(
        model,
        task1_val_loader,
        task1_val_df,
        device
    )

    val_accuracy = (
        val_metrics["accuracy"]
    )

    val_abs_eod = (
        val_metrics["abs_EOD"]
    )

    # --------------------------------------------------------
    # Checkpoint selection
    #
    # Primary:
    #   validation accuracy
    #
    # Tie-break:
    #   lower |EOD|
    #
    # This is an implementation choice because the paper does
    # not give a fully specified mathematical tie-breaking rule.
    # --------------------------------------------------------

    is_better = False

    if (
        val_accuracy
        >
        best_val_accuracy
    ):

        is_better = True

    elif (
        abs(
            val_accuracy
            -
            best_val_accuracy
        )
        < 1e-12
    ):

        if (
            val_abs_eod
            <
            best_val_abs_eod
        ):

            is_better = True

    if is_better:

        best_val_accuracy = (
            val_accuracy
        )

        best_val_abs_eod = (
            val_abs_eod
        )

        best_epoch = epoch

        best_state_dict = {
            key: value.detach().cpu().clone()
            for key, value
            in model.state_dict().items()
        }

        best_alpha_raw = (
            alpha_raw.detach()
            .cpu()
            .clone()
        )

    # --------------------------------------------------------
    # Save history
    # --------------------------------------------------------

    row = {

        "epoch": epoch,

        "train_loss":
            train_metrics["loss"],

        "train_accuracy":
            train_metrics["accuracy"],

        "val_loss":
            val_metrics["loss"],

        "val_accuracy":
            val_metrics["accuracy"],

        "val_balanced_accuracy":
            val_metrics[
                "balanced_accuracy"
            ],

        "val_macro_f1":
            val_metrics[
                "macro_f1"
            ],

        "val_accuracy_lt60":
            val_metrics[
                "accuracy_lt60"
            ],

        "val_accuracy_ge60":
            val_metrics[
                "accuracy_ge60"
            ],

        "val_EOD":
            val_metrics["EOD"],

        "val_abs_EOD":
            val_metrics["abs_EOD"],

        "val_DPR":
            val_metrics["DPR"],

        "alpha":
            train_metrics["alpha"]
    }

    history.append(row)

    print(
        f"Epoch {epoch:02d}/{WCE_EPOCHS} | "
        f"Train Loss: {train_metrics['loss']:.4f} | "
        f"Train Acc: {train_metrics['accuracy']:.4f} | "
        f"Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val |EOD|: {val_metrics['abs_EOD']:.4f} | "
        f"Alpha: {train_metrics['alpha']:.4f}"
    )


# ============================================================
# 18. RESTORE BEST VALIDATION CHECKPOINT
# ============================================================

if best_state_dict is None:

    raise RuntimeError(
        "No best WCE checkpoint was selected."
    )

model.load_state_dict(
    best_state_dict,
    strict=True
)

alpha_raw.data.copy_(
    best_alpha_raw.to(device)
)

model.eval()

best_alpha = float(
    torch.sigmoid(
        alpha_raw
    ).item()
)

print("\n" + "=" * 70)

print(
    f"Best WCE epoch: {best_epoch}"
)

print(
    f"Best validation accuracy: "
    f"{best_val_accuracy:.6f}"
)

print(
    f"Best validation |EOD|: "
    f"{best_val_abs_eod:.6f}"
)

print(
    f"Best alpha: {best_alpha:.6f}"
)

print("=" * 70)


# ============================================================
# 19. REMOVE PRUNING HOOKS
# ============================================================
#
# IMPORTANT:
#
# We remove hooks ONLY AFTER WCE training and checkpoint
# restoration.
#
# For the final checkpoint, we save the pruning masks as well.
#
# ============================================================

for handle in pruning_hooks:

    handle.remove()

print(
    "\nPruning hooks removed."
)


# ============================================================
# 20. SAVE WCE HISTORY
# ============================================================

history_df = pd.DataFrame(
    history
)

history_df.to_csv(
    HISTORY_PATH,
    index=False
)

print(
    f"Saved WCE history:\n"
    f"{HISTORY_PATH}"
)


# ============================================================
# 21. SAVE FINAL TASK-1 CHECKPOINT
# ============================================================
#
# Save:
#   model weights
#   alpha
#   pruning masks
#   pruning metadata
#
# ============================================================

serializable_masks = {}

for key, mask in pruning_masks.items():

    serializable_masks[key] = (
        mask.detach()
        .cpu()
    )


final_checkpoint = {

    "model_state_dict":
        model.state_dict(),

    "alpha_raw":
        alpha_raw.detach().cpu(),

    "alpha":
        best_alpha,

    "pruning_masks":
        serializable_masks,

    "gamma":
        GAMMA,

    "tau":
        0.7,

    "q":
        Q,

    "num_total_units":
        EXPECTED_UNITS,

    "num_pruned":
        expected_pruned,

    "num_retained":
        expected_retained,

    "best_epoch":
        best_epoch,

    "best_val_accuracy":
        best_val_accuracy,

    "best_val_abs_EOD":
        best_val_abs_eod,

    "seed":
        SEED,

    "task":
        "BKL_vs_MEL",

    "implementation_note":
        (
            "Pruning is implemented as a channel activation "
            "mask at the output of each ResNet-50 Bottleneck "
            "block, matching the representation used for "
            "Step-5 bias scoring. Exact low-level tensor "
            "surgery is an implementation decision because "
            "the paper does not fully specify the PyTorch "
            "mechanism."
        )
}

torch.save(
    final_checkpoint,
    FINAL_CHECKPOINT
)

print(
    f"Saved final Task-1 checkpoint:\n"
    f"{FINAL_CHECKPOINT}"
)


# ============================================================
# 22. SAVE CONFIGURATION
# ============================================================

config = {

    "step": 6,

    "task": "BKL_vs_MEL",

    "seed": SEED,

    "q": Q,

    "tau": 0.7,

    "gamma": GAMMA,

    "wce_epochs": WCE_EPOCHS,

    "learning_rate": LEARNING_RATE,

    "weight_decay": WEIGHT_DECAY,

    "initial_alpha": INITIAL_ALPHA,

    "best_alpha": best_alpha,

    "best_epoch": best_epoch,

    "total_units": EXPECTED_UNITS,

    "pruned_units": expected_pruned,

    "retained_units": expected_retained,

    "checkpoint_gce":
        CHECKPOINT_GCE,

    "checkpoint_final":
        FINAL_CHECKPOINT,

    "training_data":
        "Task-1 training split only",

    "validation_for_selection":
        True,

    "test_used_for_training":
        False,

    "activation_mask_pruning":
        True
}

with open(
    CONFIG_PATH,
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=2
    )

print(
    f"Saved configuration:\n"
    f"{CONFIG_PATH}"
)


# ============================================================
# 23. FINAL TASK-1 TEST EVALUATION
# ============================================================
#
# IMPORTANT:
#
# This is the FIRST time the test set is used in Step 6.
#
# It is NOT used to choose the checkpoint.
#
# ============================================================

print("\n" + "=" * 70)
print("FINAL TASK-1 TEST EVALUATION")
print("=" * 70)

# Reinstall the pruning hooks for final inference because they
# were removed after training.

final_pruning_hooks = []

for layer_name in [
    "layer1",
    "layer2",
    "layer3",
    "layer4"
]:

    layer = getattr(
        model,
        layer_name
    )

    for block_index, block in enumerate(layer):

        key = (
            f"{layer_name}.block{block_index}"
        )

        handle = block.register_forward_hook(
            make_pruning_hook(
                pruning_masks[key]
            )
        )

        final_pruning_hooks.append(
            handle
        )


test_metrics = evaluate_loader(
    model,
    task1_test_loader,
    task1_test_df,
    device
)


# Remove final inference hooks

for handle in final_pruning_hooks:

    handle.remove()


# ============================================================
# 24. PRINT TEST RESULTS
# ============================================================

print("\nTask-1 FINAL TEST RESULTS")
print("-" * 70)

print(
    f"Accuracy:              "
    f"{test_metrics['accuracy']:.6f}"
)

print(
    f"Balanced Accuracy:     "
    f"{test_metrics['balanced_accuracy']:.6f}"
)

print(
    f"Macro F1:              "
    f"{test_metrics['macro_f1']:.6f}"
)

print(
    f"Age < 60 Accuracy:     "
    f"{test_metrics['accuracy_lt60']:.6f}"
)

print(
    f"Age >= 60 Accuracy:    "
    f"{test_metrics['accuracy_ge60']:.6f}"
)

print(
    f"TPR < 60:              "
    f"{test_metrics['tpr_lt60']:.6f}"
)

print(
    f"TPR >= 60:             "
    f"{test_metrics['tpr_ge60']:.6f}"
)

print(
    f"EOD:                   "
    f"{test_metrics['EOD']:.6f}"
)

print(
    f"|EOD|:                 "
    f"{test_metrics['abs_EOD']:.6f}"
)

print(
    f"DPR:                   "
    f"{test_metrics['DPR']:.6f}"
)

print(
    f"Test Loss:             "
    f"{test_metrics['loss']:.6f}"
)


# ============================================================
# 25. SAVE TEST RESULTS
# ============================================================

test_results = {

    "task":
        "BKL_vs_MEL",

    "model":
        "Task-1 GCE + BiasPruner pruning + WCE",

    "checkpoint_epoch":
        best_epoch,

    "alpha":
        best_alpha,

    "gamma":
        GAMMA,

    "total_units":
        EXPECTED_UNITS,

    "pruned_units":
        expected_pruned,

    "retained_units":
        expected_retained,

    **{
        key: (
            float(value)
            if isinstance(
                value,
                (np.floating, np.integer)
            )
            else value
        )
        for key, value
        in test_metrics.items()
    }
}

with open(
    TEST_RESULTS_PATH,
    "w"
) as f:

    json.dump(
        test_results,
        f,
        indent=2
    )

print(
    f"\nSaved test results:\n"
    f"{TEST_RESULTS_PATH}"
)


# ============================================================
# 26. FINAL SANITY CHECKS
# ============================================================

print("\n" + "=" * 70)
print("STEP 6 SANITY CHECKS")
print("=" * 70)

# ------------------------------------------------------------
# Check 1 — alpha
# ------------------------------------------------------------

assert (
    0.0 < best_alpha < 1.0
)

print(
    "1. 0 < alpha < 1: PASS"
)

# ------------------------------------------------------------
# Check 2 — pruning count
# ------------------------------------------------------------

assert (
    expected_pruned == 9063
)

print(
    "2. 9,063 units pruned: PASS"
)

# ------------------------------------------------------------
# Check 3 — retained count
# ------------------------------------------------------------

assert (
    expected_retained == 6041
)

print(
    "3. 6,041 units retained: PASS"
)

# ------------------------------------------------------------
# Check 4 — final model
# ------------------------------------------------------------

assert os.path.exists(
    FINAL_CHECKPOINT
)

print(
    "4. Final checkpoint exists: PASS"
)

# ------------------------------------------------------------
# Check 5 — history
# ------------------------------------------------------------

assert len(history_df) == WCE_EPOCHS

print(
    "5. 20 WCE epochs recorded: PASS"
)

# ------------------------------------------------------------
# Check 6 — finite test metrics
# ------------------------------------------------------------

assert np.isfinite(
    test_metrics["accuracy"]
)

assert np.isfinite(
    test_metrics["balanced_accuracy"]
)

assert np.isfinite(
    test_metrics["macro_f1"]
)

print(
    "6. Core test metrics finite: PASS"
)

# ------------------------------------------------------------
# Check 7 — test accuracy range
# ------------------------------------------------------------

assert (
    0.0
    <=
    test_metrics["accuracy"]
    <=
    1.0
)

print(
    "7. Test accuracy valid: PASS"
)


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 70)
print("STEP 6 COMPLETE — PASS")
print("=" * 70)

print("\nFiles created:")

print(
    f"1. {FINAL_CHECKPOINT}"
)

print(
    f"2. {HISTORY_PATH}"
)

print(
    f"3. {CONFIG_PATH}"
)

print(
    f"4. {TEST_RESULTS_PATH}"
)

print("\nDo NOT start Step 7 yet.")
print(
    "We must inspect the Step-6 output before the "
    "mandatory Task-2 mechanical sanity test."
)

STEP 6 — TASK 1 PRUNING + WCE FINE-TUNING
Device: cuda
Q: 0.7
Gamma: 0.6
WCE epochs: 20
Learning rate: 0.0001
Initial alpha: 0.5

Required files and dataloaders found.

Loading Step-5 pruning plan...
Pruning plan rows: 15104
Bias-score rows: 15104

Pruning statistics:
Total units: 15104
Units marked PRUNE: 9063
Units marked RETAIN: 6041
Pruning-plan validation: PASS

Building ResNet-50...
Loading Task-1 GCE checkpoint...
GCE checkpoint loaded successfully.

Created masks for 16 Bottleneck blocks.
Registered 16 pruning hooks.

Mask verification:
Mask values: 15104
Zero/pruned: 9063
One/retained: 6041
Mask verification: PASS

Initial trainable alpha: 0.500000

Running initial pruned-model forward pass...
Sample logits shape: (32, 2)
Sample prediction shape: (32,)
Initial pruned-model forward check: PASS

STARTING WCE FINE-TUNING
Epoch 01/20 | Train Loss: 0.9089 | Train Acc: 0.6815 | Val Acc: 0.6389 | Val |EOD|: 0.0852 | Alpha: 0.4992
Epoch 02/20 | Train Loss: 0.7124 | Train Acc: 0.7483 |

NameError: name 'task1_test_df' is not defined

In [12]:
# ============================================================
# STEP 6 FIX — RECONSTRUCT TASK-1 TEST DATAFRAME
# ============================================================

print("=" * 70)
print("RECONSTRUCTING TASK-1 TEST DATAFRAME")
print("=" * 70)

# Find the master split dataframe
possible_names = [
    "split_df",
    "ham_split_df",
    "df_split",
    "ham10000_split_df"
]

found_split = None

for name in possible_names:
    if name in globals():
        found_split = globals()[name]
        print(f"Found split dataframe: {name}")
        break

if found_split is None:
    raise RuntimeError(
        "Could not find the Step-2 split dataframe. "
        "Please tell me the dataframe variable name shown in Step 2."
    )

# ------------------------------------------------------------
# Task-1 = BKL + MEL
# ------------------------------------------------------------

task1_test_df = found_split[
    (found_split["dx"].isin(["bkl", "mel"])) &
    (found_split["split"] == "test")
].copy()

# Map labels
task1_test_df["label"] = (
    task1_test_df["dx"]
    .map({
        "bkl": 0,
        "mel": 1
    })
)

task1_test_df = task1_test_df.reset_index(
    drop=True
)

print(
    f"Task-1 test samples: "
    f"{len(task1_test_df)}"
)

print("\nClass distribution:")

print(
    task1_test_df["dx"].value_counts()
)

print("\nLabel distribution:")

print(
    task1_test_df["label"].value_counts()
)

# ------------------------------------------------------------
# Verify against Step-3 test dataset
# ------------------------------------------------------------

if len(task1_test_df) != len(task1_test_loader.dataset):

    raise RuntimeError(
        f"Mismatch!\n"
        f"task1_test_df = {len(task1_test_df)}\n"
        f"test loader = {len(task1_test_loader.dataset)}"
    )

print(
    "\nDataframe / DataLoader length check: PASS"
)

print(
    "\nTask-1 test dataframe successfully reconstructed."
)

RECONSTRUCTING TASK-1 TEST DATAFRAME
Found split dataframe: split_df
Task-1 test samples: 209

Class distribution:
dx
mel    108
bkl    101
Name: count, dtype: int64

Label distribution:
label
1    108
0    101
Name: count, dtype: int64

Dataframe / DataLoader length check: PASS

Task-1 test dataframe successfully reconstructed.


In [13]:
# ============================================================
# STEP 6 — FINAL TASK-1 TEST EVALUATION ONLY
# ============================================================

print("=" * 70)
print("FINAL TASK-1 TEST EVALUATION")
print("=" * 70)

# ------------------------------------------------------------
# Load the already-trained final model
# ------------------------------------------------------------

checkpoint = torch.load(
    "/kaggle/working/BiasPruner_HAM10000/task1_final.pt",
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

model = model.to(device)
model.eval()

# Restore pruning masks from checkpoint
saved_masks = checkpoint["pruning_masks"]

pruning_masks = {
    key: value.to(device)
    for key, value in saved_masks.items()
}

# ------------------------------------------------------------
# Reinstall pruning hooks
# ------------------------------------------------------------

final_pruning_hooks = []

for layer_name in [
    "layer1",
    "layer2",
    "layer3",
    "layer4"
]:

    layer = getattr(
        model,
        layer_name
    )

    for block_index, block in enumerate(layer):

        key = (
            f"{layer_name}.block{block_index}"
        )

        handle = block.register_forward_hook(
            make_pruning_hook(
                pruning_masks[key]
            )
        )

        final_pruning_hooks.append(
            handle
        )

print(
    f"Registered {len(final_pruning_hooks)} final "
    "pruning hooks."
)

# ------------------------------------------------------------
# Evaluate TEST ONLY
# ------------------------------------------------------------

test_metrics = evaluate_loader(
    model,
    task1_test_loader,
    task1_test_df,
    device
)

# ------------------------------------------------------------
# Remove hooks
# ------------------------------------------------------------

for handle in final_pruning_hooks:
    handle.remove()

# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TASK-1 FINAL TEST RESULTS")
print("=" * 70)

print(
    f"Accuracy:              "
    f"{test_metrics['accuracy']:.6f}"
)

print(
    f"Balanced Accuracy:     "
    f"{test_metrics['balanced_accuracy']:.6f}"
)

print(
    f"Macro F1:              "
    f"{test_metrics['macro_f1']:.6f}"
)

print(
    f"Age < 60 Accuracy:     "
    f"{test_metrics['accuracy_lt60']:.6f}"
)

print(
    f"Age >= 60 Accuracy:    "
    f"{test_metrics['accuracy_ge60']:.6f}"
)

print(
    f"TPR < 60:              "
    f"{test_metrics['tpr_lt60']:.6f}"
)

print(
    f"TPR >= 60:             "
    f"{test_metrics['tpr_ge60']:.6f}"
)

print(
    f"EOD:                   "
    f"{test_metrics['EOD']:.6f}"
)

print(
    f"|EOD|:                 "
    f"{test_metrics['abs_EOD']:.6f}"
)

print(
    f"DPR:                   "
    f"{test_metrics['DPR']:.6f}"
)

print(
    f"Test Loss:             "
    f"{test_metrics['loss']:.6f}"
)

# ------------------------------------------------------------
# Save final results
# ------------------------------------------------------------

test_results_path = (
    "/kaggle/working/BiasPruner_HAM10000/"
    "task1_test_results.json"
)

test_results = {
    "task": "BKL_vs_MEL",
    "model": "Task1_GCE_BiasPruner_WCE",
    "best_wce_epoch": int(
        checkpoint["best_epoch"]
    ),
    "alpha": float(
        checkpoint["alpha"]
    ),
    "gamma": float(
        checkpoint["gamma"]
    ),
    "total_units": int(
        checkpoint["num_total_units"]
    ),
    "pruned_units": int(
        checkpoint["num_pruned"]
    ),
    "retained_units": int(
        checkpoint["num_retained"]
    ),
    **{
        k: float(v)
        for k, v in test_metrics.items()
    }
}

with open(
    test_results_path,
    "w"
) as f:

    json.dump(
        test_results,
        f,
        indent=2
    )

print(
    f"\nSaved:\n{test_results_path}"
)

print("\n" + "=" * 70)
print("STEP 6 FINAL EVALUATION COMPLETE")
print("=" * 70)

FINAL TASK-1 TEST EVALUATION
Registered 16 final pruning hooks.

TASK-1 FINAL TEST RESULTS
Accuracy:              0.808612
Balanced Accuracy:     0.808718
Macro F1:              0.808503
Age < 60 Accuracy:     0.837838
Age >= 60 Accuracy:    0.791045
TPR < 60:              0.812500
TPR >= 60:             0.800000
EOD:                   0.012500
|EOD|:                 0.012500
DPR:                   1.188345
Test Loss:             0.437562

Saved:
/kaggle/working/BiasPruner_HAM10000/task1_test_results.json

STEP 6 FINAL EVALUATION COMPLETE


In [14]:
# ============================================================
# STEP 7 — TASK-2 MECHANICAL SANITY TEST
# ============================================================
#
# Task 1:
#   BKL  -> 0
#   MEL  -> 1
#
# Task 2:
#   BCC   -> 0
#   AKIEC -> 1
#
# Purpose:
#   Verify that the continual-learning implementation can:
#
#   1. Keep Task-1 protected parameters unchanged
#   2. Allow Task-2 free parameters to learn
#   3. Keep Task-1 protected representation stable
#   4. Preserve tensor shapes
#   5. Preserve residual connections
#   6. Avoid collapsed Task-2 predictions
#   7. Produce ~zero gradient on protected parameters
#   8. Produce non-zero gradient on free parameters
#
# IMPORTANT:
#   This cell does NOT perform full Task-2 training.
#
# ============================================================

import os
import json
import copy
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50


# ============================================================
# 0. CONFIGURATION
# ============================================================

SEED = 42

TASK1_CHECKPOINT = (
    "/kaggle/working/BiasPruner_HAM10000/"
    "task1_final.pt"
)

PRUNING_PLAN_PATH = (
    "/kaggle/working/BiasPruner_HAM10000/"
    "task1_pruning_plan.csv"
)

OUTPUT_DIR = (
    "/kaggle/working/BiasPruner_HAM10000"
)

SANITY_RESULTS_PATH = os.path.join(
    OUTPUT_DIR,
    "task2_mechanical_sanity_results.json"
)

# Small learning rate only for the mechanical test.
# This is NOT Task-2 training.
SANITY_LR = 1e-4


# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("STEP 7 — TASK-2 MECHANICAL SANITY TEST")
print("=" * 70)

print(f"Device: {device}")


# ============================================================
# 2. CHECK REQUIRED FILES / OBJECTS
# ============================================================

if not os.path.exists(
    TASK1_CHECKPOINT
):

    raise FileNotFoundError(
        f"Task-1 final checkpoint not found:\n"
        f"{TASK1_CHECKPOINT}"
    )

if not os.path.exists(
    PRUNING_PLAN_PATH
):

    raise FileNotFoundError(
        f"Pruning plan not found:\n"
        f"{PRUNING_PLAN_PATH}"
    )

required_objects = [
    "task1_test_loader",
    "task2_train_loader",
    "task2_val_loader",
    "task2_test_loader"
]

for name in required_objects:

    if name not in globals():

        raise RuntimeError(
            f"Required object '{name}' is missing.\n"
            "Please run Step 3 first."
        )

print(
    "\nRequired checkpoint, pruning plan, "
    "and dataloaders found."
)


# ============================================================
# 3. LOAD TASK-1 FINAL MODEL
# ============================================================

print("\nLoading Task-1 final model...")

checkpoint = torch.load(
    TASK1_CHECKPOINT,
    map_location=device
)

model = resnet50(
    weights=None
)

model.fc = nn.Linear(
    model.fc.in_features,
    2
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

model = model.to(device)

print(
    "Task-1 final model loaded successfully."
)


# ============================================================
# 4. LOAD TASK-1 PRUNING MASK
# ============================================================

if "pruning_masks" not in checkpoint:

    raise RuntimeError(
        "Task-1 checkpoint does not contain "
        "pruning masks."
    )

saved_masks = checkpoint[
    "pruning_masks"
]

pruning_masks = {

    key: value.to(device)
    for key, value
    in saved_masks.items()
}

print(
    f"Loaded {len(pruning_masks)} "
    "Task-1 pruning masks."
)


# ============================================================
# 5. BUILD TASK-1 PROTECTED / FREE UNIT MAP
# ============================================================
#
# Task-1 pruning:
#
#   mask = 1
#       retained / protected
#
#   mask = 0
#       pruned / free
#
# The paper's continual transfer mechanism allows a new task
# to use previously assigned debiased subnetworks together with
# free units.
#
# Therefore:
#
#   Task-2 representation =
#       Task-1 protected units
#       +
#       free units
#
# Previous Task-1 protected parameters themselves must remain
# frozen.
#
# ============================================================

protected_units = 0
free_units = 0

for key, mask in pruning_masks.items():

    protected_units += int(
        (mask == 1).sum().item()
    )

    free_units += int(
        (mask == 0).sum().item()
    )

total_units = (
    protected_units
    +
    free_units
)

print("\nTask-1 unit ownership:")
print(
    f"Protected units: {protected_units}"
)
print(
    f"Free units:      {free_units}"
)
print(
    f"Total units:     {total_units}"
)

assert total_units == 15104
assert protected_units == 6041
assert free_units == 9063

print(
    "Unit ownership check: PASS"
)


# ============================================================
# 6. CREATE TASK-2 MODEL
# ============================================================
#
# We start from the Task-1 final model.
#
# Task-2 has:
#
#   BCC   -> 0
#   AKIEC -> 1
#
# The final classifier must be trainable for Task 2.
#
# ============================================================

task2_model = copy.deepcopy(
    model
)

task2_model = task2_model.to(device)

print(
    "\nTask-2 model created from Task-1 final model."
)


# ============================================================
# 7. PROTECTED PARAMETER DEFINITIONS
# ============================================================
#
# IMPORTANT:
#
# A unit mask alone does not prove parameter immutability.
#
# For this mechanical test we explicitly create parameter-level
# gradient masks for convolutional output channels.
#
# For each Bottleneck:
#
#   conv1: output channels
#   conv2: output channels
#   conv3: output channels
#
# We associate the output-channel ownership with the
# corresponding Bottleneck output representation.
#
# The residual/downsample parameters are NOT modified here.
#
# ============================================================

parameter_gradient_masks = {}


def add_conv_gradient_mask(
    parameter,
    channel_mask
):

    """
    Register a gradient hook so that protected output
    channels cannot receive gradients.
    """

    # channel_mask shape:
    # [C]

    def gradient_hook(grad):

        mask = channel_mask.to(
            grad.device
        )

        # Weight tensor:
        # [C_out, ...]
        #
        # Bias tensor:
        # [C_out]

        if grad.ndim == 1:

            return (
                grad * mask
            )

        else:

            view_shape = [
                -1
            ] + [
                1
            ] * (
                grad.ndim - 1
            )

            return (
                grad
                *
                mask.view(
                    *view_shape
                )
            )

    return parameter.register_hook(
        gradient_hook
    )


gradient_hooks = []

protected_parameter_names = []
free_parameter_names = []


# ============================================================
# 8. REGISTER GRADIENT MASKS ON BOTTLENECK CONVS
# ============================================================

for layer_name in [
    "layer1",
    "layer2",
    "layer3",
    "layer4"
]:

    layer = getattr(
        task2_model,
        layer_name
    )

    for block_index, block in enumerate(layer):

        key = (
            f"{layer_name}.block{block_index}"
        )

        unit_mask = pruning_masks[key]

        # ----------------------------------------------------
        # conv1
        # ----------------------------------------------------
        #
        # conv1 output channels correspond to the internal
        # bottleneck representation.
        #
        # The channel ownership is mapped proportionally to
        # the Bottleneck output ownership only when dimensions
        # match.
        #
        # For ResNet-50 torchvision, conv3 output channels are
        # the exact unit representation used by Step 5.
        #
        # Therefore conv3 gets the exact mask.
        #
        # conv1/conv2 remain ordinary trainable parameters,
        # while the exact protected representation is enforced
        # at conv3.
        #
        # ----------------------------------------------------

        conv3 = block.conv3

        # Exact output-channel mask
        hook = add_conv_gradient_mask(
            conv3.weight,
            unit_mask
        )

        gradient_hooks.append(
            hook
        )

        protected_parameter_names.append(
            f"{key}.conv3.weight"
        )

        if conv3.bias is not None:

            hook = add_conv_gradient_mask(
                conv3.bias,
                unit_mask
            )

            gradient_hooks.append(
                hook
            )

            protected_parameter_names.append(
                f"{key}.conv3.bias"
            )


# ============================================================
# 9. FREE-UNIT GRADIENT TARGET
# ============================================================
#
# To verify that the free units really can learn, we will
# inspect the conv3 parameters corresponding to mask = 0.
#
# ============================================================

free_parameter_names = []

for layer_name in [
    "layer1",
    "layer2",
    "layer3",
    "layer4"
]:

    layer = getattr(
        task2_model,
        layer_name
    )

    for block_index, block in enumerate(layer):

        key = (
            f"{layer_name}.block{block_index}"
        )

        mask = pruning_masks[key]

        if (
            (mask == 0).sum().item()
            > 0
        ):

            free_parameter_names.append(
                f"{key}.conv3.weight"
            )

            if block.conv3.bias is not None:

                free_parameter_names.append(
                    f"{key}.conv3.bias"
                )

print(
    f"\nProtected conv3 parameter groups: "
    f"{len(protected_parameter_names)}"
)

print(
    f"Free conv3 parameter groups: "
    f"{len(free_parameter_names)}"
)


# ============================================================
# 10. SNAPSHOT TASK-1 PROTECTED PARAMETERS
# ============================================================

protected_parameter_snapshot = {}

for name, parameter in (
    task2_model.named_parameters()
):

    if name in protected_parameter_names:

        protected_parameter_snapshot[name] = (
            parameter.detach()
            .cpu()
            .clone()
        )

print(
    f"Saved {len(protected_parameter_snapshot)} "
    "protected parameter snapshots."
)


# ============================================================
# 11. SNAPSHOT TASK-1 REPRESENTATION
# ============================================================
#
# We use a small fixed batch from Task-1 test data.
#
# This is ONLY a mechanical stability test.
# We do not use labels to tune anything.
#
# ============================================================

task1_test_images, task1_test_labels = next(
    iter(task1_test_loader)
)

task1_test_images = (
    task1_test_images.to(device)
)


# ------------------------------------------------------------
# Capture Task-1 representation
# ------------------------------------------------------------

representation_before = {}

representation_hooks = []


def make_representation_hook(
    key
):

    def hook(
        module,
        inputs,
        output
    ):

        representation_before[key] = (
            output.detach()
            .cpu()
            .clone()
        )

    return hook


for layer_name in [
    "layer1",
    "layer2",
    "layer3",
    "layer4"
]:

    layer = getattr(
        task2_model,
        layer_name
    )

    for block_index, block in enumerate(layer):

        key = (
            f"{layer_name}.block{block_index}"
        )

        handle = block.register_forward_hook(
            make_representation_hook(
                key
            )
        )

        representation_hooks.append(
            handle
        )


task2_model.eval()

with torch.no_grad():

    _ = task2_model(
        task1_test_images
    )


for handle in representation_hooks:

    handle.remove()


print(
    "\nTask-1 protected representation snapshot: PASS"
)


# ============================================================
# 12. TASK-2 FORWARD SHAPE TEST
# ============================================================

print("\nRunning Task-2 forward test...")

task2_images, task2_labels = next(
    iter(task2_train_loader)
)

task2_images = task2_images.to(
    device
)

task2_labels = task2_labels.to(
    device
)

task2_model.eval()

with torch.no_grad():

    task2_logits_before = task2_model(
        task2_images
    )

print(
    f"Task-2 input shape: "
    f"{tuple(task2_images.shape)}"
)

print(
    f"Task-2 output shape: "
    f"{tuple(task2_logits_before.shape)}"
)

assert (
    task2_logits_before.shape[0]
    ==
    task2_images.shape[0]
)

assert (
    task2_logits_before.shape[1]
    ==
    2
)

assert torch.isfinite(
    task2_logits_before
).all()

print(
    "Tensor-shape / finite-output check: PASS"
)


# ============================================================
# 13. TASK-2 PREDICTION COLLAPSE TEST
# ============================================================

task2_predictions_before = (
    torch.argmax(
        task2_logits_before,
        dim=1
    )
)

class0_count = int(
    (
        task2_predictions_before == 0
    ).sum().item()
)

class1_count = int(
    (
        task2_predictions_before == 1
    ).sum().item()
)

total_predictions = (
    class0_count
    +
    class1_count
)

class0_rate = (
    class0_count
    /
    total_predictions
)

class1_rate = (
    class1_count
    /
    total_predictions
)

print("\nTask-2 prediction distribution BEFORE update:")
print(
    f"Class 0: {class0_count} "
    f"({class0_rate:.3f})"
)

print(
    f"Class 1: {class1_count} "
    f"({class1_rate:.3f})"
)

# Collapse means every prediction is one class.
prediction_collapsed = (
    class0_count == 0
    or
    class1_count == 0
)

if prediction_collapsed:

    print(
        "WARNING: Task-2 initial predictions "
        "are collapsed."
    )

else:

    print(
        "Task-2 prediction collapse check: PASS"
    )


# ============================================================
# 14. GRADIENT TEST
# ============================================================
#
# One forward/backward pass using Task-2 data.
#
# We use ordinary cross entropy here.
# This is ONLY to inspect gradient flow.
#
# ============================================================

task2_model.train()

gradient_optimizer = torch.optim.Adam(
    task2_model.parameters(),
    lr=SANITY_LR
)

gradient_optimizer.zero_grad(
    set_to_none=True
)

task2_logits = task2_model(
    task2_images
)

sanity_loss = F.cross_entropy(
    task2_logits,
    task2_labels
)

sanity_loss.backward()


# ============================================================
# 15. MEASURE PROTECTED / FREE GRADIENT NORMS
# ============================================================

protected_gradient_squared = 0.0
free_gradient_squared = 0.0

protected_gradient_count = 0
free_gradient_count = 0

for name, parameter in (
    task2_model.named_parameters()
):

    if parameter.grad is None:
        continue

    grad_norm_sq = (
        parameter.grad.detach()
        .float()
        .pow(2)
        .sum()
        .item()
    )

    if name in protected_parameter_names:

        protected_gradient_squared += (
            grad_norm_sq
        )

        protected_gradient_count += 1

    elif name in free_parameter_names:

        free_gradient_squared += (
            grad_norm_sq
        )

        free_gradient_count += 1


protected_gradient_norm = math.sqrt(
    protected_gradient_squared
)

free_gradient_norm = math.sqrt(
    free_gradient_squared
)

print("\n" + "=" * 70)
print("GRADIENT FLOW")
print("=" * 70)

print(
    f"Protected gradient norm: "
    f"{protected_gradient_norm:.12e}"
)

print(
    f"Free gradient norm:      "
    f"{free_gradient_norm:.12e}"
)

print(
    f"Protected parameter groups with gradients: "
    f"{protected_gradient_count}"
)

print(
    f"Free parameter groups with gradients: "
    f"{free_gradient_count}"
)


# ============================================================
# 16. GRADIENT CHECKS
# ============================================================

protected_gradient_ok = (
    protected_gradient_norm
    <=
    1e-10
)

free_gradient_ok = (
    free_gradient_norm
    >
    1e-12
)

print(
    "\nProtected gradient ≈ 0:",
    "PASS" if protected_gradient_ok
    else "FAIL"
)

print(
    "Free gradient > 0:",
    "PASS" if free_gradient_ok
    else "FAIL"
)


# ============================================================
# 17. ONE OPTIMIZER UPDATE
# ============================================================
#
# This is NOT Task-2 training.
#
# It is exactly ONE optimizer update used to prove:
#
#   protected parameters don't move
#   free parameters can move
#
# ============================================================

# Snapshot free parameters before update

free_parameter_snapshot = {}

for name, parameter in (
    task2_model.named_parameters()
):

    if name in free_parameter_names:

        free_parameter_snapshot[name] = (
            parameter.detach()
            .cpu()
            .clone()
        )


# Snapshot ALL parameters before update

all_parameter_snapshot = {

    name:
        parameter.detach()
        .cpu()
        .clone()

    for name, parameter
    in task2_model.named_parameters()
}


gradient_optimizer.step()


# ============================================================
# 18. MEASURE PARAMETER CHANGES
# ============================================================

protected_max_change = 0.0
free_max_change = 0.0
all_max_change = 0.0

free_changed_count = 0
protected_changed_count = 0

for name, parameter in (
    task2_model.named_parameters()
):

    before = all_parameter_snapshot[
        name
    ]

    after = (
        parameter.detach()
        .cpu()
    )

    max_change = float(
        torch.max(
            torch.abs(
                after - before
            )
        ).item()
    )

    all_max_change = max(
        all_max_change,
        max_change
    )

    if name in protected_parameter_names:

        protected_max_change = max(
            protected_max_change,
            max_change
        )

        if max_change > 1e-12:

            protected_changed_count += 1

    if name in free_parameter_names:

        free_max_change = max(
            free_max_change,
            max_change
        )

        if max_change > 1e-12:

            free_changed_count += 1


print("\n" + "=" * 70)
print("PARAMETER IMMUTABILITY TEST")
print("=" * 70)

print(
    f"Protected max parameter change: "
    f"{protected_max_change:.12e}"
)

print(
    f"Free max parameter change:      "
    f"{free_max_change:.12e}"
)

print(
    f"Protected parameter groups changed: "
    f"{protected_changed_count}"
)

print(
    f"Free parameter groups changed: "
    f"{free_changed_count}"
)


protected_parameters_ok = (
    protected_max_change
    <=
    1e-12
)

free_parameters_ok = (
    free_max_change
    >
    1e-12
)

print(
    "\nTask-1 protected parameters unchanged:",
    "PASS" if protected_parameters_ok
    else "FAIL"
)

print(
    "Task-2 free parameters changed:",
    "PASS" if free_parameters_ok
    else "FAIL"
)


# ============================================================
# 19. RESIDUAL / REPRESENTATION STABILITY TEST
# ============================================================
#
# We now compare the protected Task-1 representation after the
# one Task-2 update.
#
# This is a mechanical test that the Task-1 representation has
# not moved due to the Task-2 update.
#
# ============================================================

representation_after = {}

representation_hooks = []


def make_after_hook(
    key
):

    def hook(
        module,
        inputs,
        output
    ):

        representation_after[key] = (
            output.detach()
            .cpu()
            .clone()
        )

    return hook


for layer_name in [
    "layer1",
    "layer2",
    "layer3",
    "layer4"
]:

    layer = getattr(
        task2_model,
        layer_name
    )

    for block_index, block in enumerate(layer):

        key = (
            f"{layer_name}.block{block_index}"
        )

        handle = block.register_forward_hook(
            make_after_hook(
                key
            )
        )

        representation_hooks.append(
            handle
        )


task2_model.eval()

with torch.no_grad():

    _ = task2_model(
        task1_test_images
    )


for handle in representation_hooks:

    handle.remove()


# ------------------------------------------------------------
# Compare representations
# ------------------------------------------------------------

max_representation_difference = 0.0

for key in representation_before:

    before = representation_before[
        key
    ]

    after = representation_after[
        key
    ]

    difference = float(
        torch.max(
            torch.abs(
                before - after
            )
        ).item()
    )

    max_representation_difference = max(
        max_representation_difference,
        difference
    )


representation_stable = (
    max_representation_difference
    <=
    1e-10
)

print("\n" + "=" * 70)
print("TASK-1 REPRESENTATION STABILITY")
print("=" * 70)

print(
    f"Maximum representation difference: "
    f"{max_representation_difference:.12e}"
)

print(
    "Protected representation stable:",
    "PASS" if representation_stable
    else "FAIL"
)


# ============================================================
# 20. FINAL SANITY DECISION
# ============================================================

print("\n" + "=" * 70)
print("STEP 7 FINAL SANITY REPORT")
print("=" * 70)

checks = {

    "task1_parameters_unchanged":
        protected_parameters_ok,

    "task2_free_parameters_changed":
        free_parameters_ok,

    "task1_representation_stable":
        representation_stable,

    "tensor_shapes_valid":
        (
            task2_logits_before.shape[1]
            ==
            2
        ),

    "residual_forward_valid":
        torch.isfinite(
            task2_logits_before
        ).all().item(),

    "task2_predictions_not_collapsed":
        not prediction_collapsed,

    "protected_gradient_zero":
        protected_gradient_ok,

    "free_gradient_nonzero":
        free_gradient_ok
}


for name, result in checks.items():

    print(
        f"{name}: "
        f"{'PASS' if result else 'FAIL'}"
    )


all_pass = all(
    checks.values()
)

# ============================================================
# 21. SAVE RESULTS
# ============================================================

results = {

    "step": 7,

    "task1":
        "BKL_vs_MEL",

    "task2":
        "BCC_vs_AKIEC",

    "protected_units":
        protected_units,

    "free_units":
        free_units,

    "total_units":
        total_units,

    "protected_gradient_norm":
        protected_gradient_norm,

    "free_gradient_norm":
        free_gradient_norm,

    "protected_max_parameter_change":
        protected_max_change,

    "free_max_parameter_change":
        free_max_change,

    "protected_changed_count":
        protected_changed_count,

    "free_changed_count":
        free_changed_count,

    "max_representation_difference":
        max_representation_difference,

    "task2_class0_predictions":
        class0_count,

    "task2_class1_predictions":
        class1_count,

    "checks":
        {
            key: bool(value)
            for key, value
            in checks.items()
        },

    "all_pass":
        bool(all_pass)
}

with open(
    SANITY_RESULTS_PATH,
    "w"
) as f:

    json.dump(
        results,
        f,
        indent=2
    )


# ============================================================
# 22. CLEAN UP GRADIENT HOOKS
# ============================================================

for handle in gradient_hooks:

    handle.remove()


# ============================================================
# 23. FINAL DECISION
# ============================================================

print(
    f"\nSaved sanity results:\n"
    f"{SANITY_RESULTS_PATH}"
)

print("\n" + "=" * 70)

if all_pass:

    print(
        "STEP 7 COMPLETE — PASS"
    )

    print(
        "\nAll mandatory mechanical checks passed."
    )

    print(
        "Task-2 training is now mechanically permitted."
    )

else:

    print(
        "STEP 7 — FAIL"
    )

    print(
        "\nDO NOT START TASK-2 TRAINING."
    )

    failed = [
        name
        for name, result
        in checks.items()
        if not result
    ]

    print("\nFailed checks:")

    for item in failed:

        print(
            f"  - {item}"
        )

print("=" * 70)

STEP 7 — TASK-2 MECHANICAL SANITY TEST
Device: cuda

Required checkpoint, pruning plan, and dataloaders found.

Loading Task-1 final model...
Task-1 final model loaded successfully.
Loaded 16 Task-1 pruning masks.

Task-1 unit ownership:
Protected units: 6041
Free units:      9063
Total units:     15104
Unit ownership check: PASS

Task-2 model created from Task-1 final model.

Protected conv3 parameter groups: 16
Free conv3 parameter groups: 16
Saved 0 protected parameter snapshots.

Task-1 protected representation snapshot: PASS

Running Task-2 forward test...
Task-2 input shape: (32, 3, 224, 224)
Task-2 output shape: (32, 2)
Tensor-shape / finite-output check: PASS

Task-2 prediction distribution BEFORE update:
Class 0: 0 (0.000)
Class 1: 32 (1.000)

GRADIENT FLOW
Protected gradient norm: 0.000000000000e+00
Free gradient norm:      0.000000000000e+00
Protected parameter groups with gradients: 0
Free parameter groups with gradients: 0

Protected gradient ≈ 0: PASS
Free gradient > 0: F

In [18]:
# ============================================================
# STEP 7 — TASK-2 MECHANICAL SANITY TEST v5
# ============================================================
#
# PURPOSE
# -------
# Verify the continual-learning mechanism BEFORE Task-2 GCE.
#
# Task-1 protected subnet:
#       FROZEN + ACTIVE
#
# Task-2 free subnet:
#       TRAINABLE + ACTIVE
#
# Task-2 representation:
#
#       protected Task-1 units
#                  +
#       free Task-2 units
#
# IMPORTANT
# ---------
# The Step-5 pruning CSV uses:
#
#       prune = 0  --> retained / protected
#       prune = 1  --> pruned / free
#
# Internally we use:
#
#       protected_mask = 1
#       free_mask      = 0
#
# This is a fixed-shape implementation decision because
# the paper describes the continual-transfer mechanism but
# does not specify the exact PyTorch tensor-surgery mechanism.
#
# DO NOT PROCEED TO STEP 8 IF STEP 7 FAILS.
# ============================================================


# ============================================================
# IMPORTS
# ============================================================

import os
import json
import copy
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torchvision.models import resnet50


# ============================================================
# 7.1 CONFIGURATION
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

WORK_DIR = "/kaggle/working"

PROJECT_DIR = os.path.join(
    WORK_DIR,
    "BiasPruner_HAM10000"
)

TASK1_FINAL_CKPT = os.path.join(
    PROJECT_DIR,
    "task1_final.pt"
)

PRUNING_PLAN_PATH = os.path.join(
    PROJECT_DIR,
    "task1_pruning_plan.csv"
)

STEP7_RESULT_PATH = os.path.join(
    PROJECT_DIR,
    "task2_mechanical_sanity_v5.json"
)

NUM_CLASSES = 2

print("=" * 75)
print("STEP 7 — TASK-2 MECHANICAL SANITY TEST v5")
print("=" * 75)

print(
    f"Device: {DEVICE}"
)

print(
    f"Task-1 checkpoint: {TASK1_FINAL_CKPT}"
)

print(
    f"Pruning plan:      {PRUNING_PLAN_PATH}"
)


# ============================================================
# 7.2 CHECK REQUIRED FILES
# ============================================================

assert os.path.exists(
    TASK1_FINAL_CKPT
), (
    "Task-1 checkpoint not found:\n"
    f"{TASK1_FINAL_CKPT}"
)

assert os.path.exists(
    PRUNING_PLAN_PATH
), (
    "Pruning plan not found:\n"
    f"{PRUNING_PLAN_PATH}"
)

print(
    "\nRequired files: PASS"
)


# ============================================================
# 7.3 LOAD TASK-1 CHECKPOINT
# ============================================================

checkpoint = torch.load(
    TASK1_FINAL_CKPT,
    map_location="cpu"
)

print(
    "\nTask-1 checkpoint keys:"
)

print(
    checkpoint.keys()
)


# ============================================================
# 7.4 BUILD RESNET-50 BINARY MODEL
# ============================================================

def build_resnet50_binary():

    model = resnet50(
        weights=None
    )

    model.fc = nn.Linear(
        model.fc.in_features,
        NUM_CLASSES
    )

    return model


# ============================================================
# 7.5 EXTRACT STATE DICT
# ============================================================

if "model_state_dict" in checkpoint:

    state_dict = checkpoint[
        "model_state_dict"
    ]

elif "state_dict" in checkpoint:

    state_dict = checkpoint[
        "state_dict"
    ]

else:

    state_dict = checkpoint


# ============================================================
# 7.6 LOAD TASK-1 REFERENCE MODEL
# ============================================================

task1_reference = (
    build_resnet50_binary()
)

task1_reference.load_state_dict(
    state_dict,
    strict=True
)

task1_reference = (
    task1_reference.to(DEVICE)
)

print(
    "\nTask-1 model loaded successfully."
)

print(
    "Task-1 parameters:",
    sum(
        p.numel()
        for p in task1_reference.parameters()
    )
)


# ============================================================
# 7.7 LOAD PRUNING PLAN
# ============================================================

pruning_df = pd.read_csv(
    PRUNING_PLAN_PATH
)

print(
    "\nPruning-plan columns:"
)

print(
    pruning_df.columns.tolist()
)

print(
    "Pruning-plan shape:",
    pruning_df.shape
)


# ============================================================
# 7.8 EXPECTED RESNET-50 UNIT COUNTS
# ============================================================

EXPECTED_UNITS = {

    "layer1": 3 * 256,

    "layer2": 4 * 512,

    "layer3": 6 * 1024,

    "layer4": 3 * 2048,
}

EXPECTED_TOTAL_UNITS = sum(
    EXPECTED_UNITS.values()
)

print(
    "\nExpected total units:",
    EXPECTED_TOTAL_UNITS
)

assert (
    EXPECTED_TOTAL_UNITS == 15104
)


# ============================================================
# 7.9 VERIFY PRUNING PLAN FORMAT
# ============================================================

required_columns = [

    "global_unit_id",

    "layer",

    "block_index",

    "channel",

    "bias_score",

    "bias_rank",

    "prune",

    "status",
]

missing_columns = [

    c
    for c in required_columns
    if c not in pruning_df.columns
]

assert len(
    missing_columns
) == 0, (
    "Missing pruning-plan columns: "
    f"{missing_columns}"
)

print(
    "\nPruning-plan format: PASS"
)


# ============================================================
# 7.10 VERIFY PRUNE COLUMN
# ============================================================

unique_prune_values = sorted(
    pruning_df[
        "prune"
    ]
    .dropna()
    .unique()
    .tolist()
)

print(
    "\nUnique values in 'prune':",
    unique_prune_values
)

assert set(
    unique_prune_values
).issubset(
    {0, 1}
), (
    "The prune column must contain only 0 and 1."
)


# ============================================================
# 7.11 CONVERT PRUNE -> PROTECTED/FREE OWNERSHIP
# ============================================================
#
# CSV:
#
#       prune=0 -> protected
#       prune=1 -> free
#
# Internal:
#
#       protected=1
#       free=0
#
# Therefore:
#
#       protected_mask = 1 - prune
# ============================================================

block_masks = {}


for _, row in pruning_df.iterrows():

    # --------------------------------------------------------
    # Normalize layer name
    # --------------------------------------------------------

    layer = str(
        row["layer"]
    ).lower().strip()

    if layer in [
        "1",
        "layer_1"
    ]:

        layer = "layer1"

    elif layer in [
        "2",
        "layer_2"
    ]:

        layer = "layer2"

    elif layer in [
        "3",
        "layer_3"
    ]:

        layer = "layer3"

    elif layer in [
        "4",
        "layer_4"
    ]:

        layer = "layer4"


    # --------------------------------------------------------
    # Block and channel
    # --------------------------------------------------------

    block = int(
        row["block_index"]
    )

    channel = int(
        row["channel"]
    )


    # --------------------------------------------------------
    # Pruning decision
    # --------------------------------------------------------

    prune_value = int(
        row["prune"]
    )


    # --------------------------------------------------------
    # Convert to protected representation
    # --------------------------------------------------------

    protected_value = (
        1 - prune_value
    )


    key = (
        layer,
        block
    )


    if key not in block_masks:

        block_masks[key] = {}


    block_masks[key][
        channel
    ] = protected_value


print(
    "\nOwnership conversion completed."
)

print(
    "Internal meaning:"
)

print(
    "    1 = protected / retained"
)

print(
    "    0 = free / pruned"
)


# ============================================================
# 7.12 RESNET LAYER STRUCTURE
# ============================================================

layer_block_counts = {

    "layer1": 3,

    "layer2": 4,

    "layer3": 6,

    "layer4": 3,
}


layer_channels = {

    "layer1": 256,

    "layer2": 512,

    "layer3": 1024,

    "layer4": 2048,
}


# ============================================================
# 7.13 HANDLE 0-BASED / 1-BASED BLOCK INDEX
# ============================================================

for layer in [
    "layer1",
    "layer2",
    "layer3",
    "layer4",
]:

    n_blocks = (
        layer_block_counts[layer]
    )

    for block_idx in range(
        n_blocks
    ):

        zero_based_key = (
            layer,
            block_idx
        )

        one_based_key = (
            layer,
            block_idx + 1
        )

        if (
            zero_based_key
            not in block_masks
            and
            one_based_key
            in block_masks
        ):

            block_masks[
                zero_based_key
            ] = block_masks.pop(
                one_based_key
            )


# ============================================================
# 7.14 VALIDATE EVERY BLOCK
# ============================================================

total_protected = 0
total_free = 0

print(
    "\nPer-block ownership:"
)


for layer in [
    "layer1",
    "layer2",
    "layer3",
    "layer4",
]:

    n_blocks = (
        layer_block_counts[layer]
    )

    channels = (
        layer_channels[layer]
    )


    for block_idx in range(
        n_blocks
    ):

        key = (
            layer,
            block_idx
        )

        assert key in block_masks, (
            f"Missing mask for {key}"
        )


        mask_dict = (
            block_masks[key]
        )


        assert len(
            mask_dict
        ) == channels, (
            f"{key}: expected "
            f"{channels} channels, "
            f"found {len(mask_dict)}"
        )


        expected_channels = set(
            range(channels)
        )

        actual_channels = set(
            mask_dict.keys()
        )


        assert (
            actual_channels
            ==
            expected_channels
        ), (
            f"{key}: channel indices "
            "do not match expected range."
        )


        protected_count = sum(

            1
            for v in mask_dict.values()
            if v == 1

        )


        free_count = sum(

            1
            for v in mask_dict.values()
            if v == 0

        )


        assert (
            protected_count
            +
            free_count
            ==
            channels
        )


        total_protected += (
            protected_count
        )

        total_free += (
            free_count
        )


        print(
            f"  {layer} block {block_idx}: "
            f"protected={protected_count:4d}, "
            f"free={free_count:4d}, "
            f"total={channels:4d}"
        )


# ============================================================
# 7.15 GLOBAL OWNERSHIP CHECK
# ============================================================

print(
    "\nGlobal ownership:"
)

print(
    "Protected units:",
    total_protected
)

print(
    "Free units:",
    total_free
)

print(
    "Total units:",
    total_protected + total_free
)


assert (
    total_protected == 6041
), (
    f"Expected 6041 protected units, "
    f"got {total_protected}"
)


assert (
    total_free == 9063
), (
    f"Expected 9063 free units, "
    f"got {total_free}"
)


assert (
    total_protected
    +
    total_free
    ==
    15104
)


print(
    "PASS — protected/free ownership "
    "matches Step 5."
)


# ============================================================
# 7.16 CREATE TORCH MASKS
# ============================================================

torch_block_masks = {}


for (
    layer,
    block_idx
), mask_dict in block_masks.items():

    channels = (
        layer_channels[layer]
    )

    mask = torch.tensor(

        [
            mask_dict[i]
            for i in range(channels)
        ],

        dtype=torch.float32,

        device=DEVICE
    )


    torch_block_masks[
        (layer, block_idx)
    ] = mask


print(
    "\nTorch masks created:",
    len(torch_block_masks)
)

assert (
    len(torch_block_masks)
    == 16
)


# ============================================================
# 7.17 CONTINUAL RESNET-50
# ============================================================

class ContinualResNet50(
    nn.Module
):


    def __init__(
        self,
        protected_model,
        free_model,
        block_masks
    ):

        super().__init__()


        self.protected_model = (
            protected_model
        )


        self.free_model = (
            free_model
        )


        self.block_masks = (
            block_masks
        )


        # ----------------------------------------------------
        # TASK-1 PROTECTED BRANCH
        # ----------------------------------------------------
        #
        # Completely frozen.
        # ----------------------------------------------------

        for p in (
            self.protected_model.parameters()
        ):

            p.requires_grad = False


        # ----------------------------------------------------
        # SHARED/FROZEN STEM
        # ----------------------------------------------------
        #
        # Task-specific unit ownership starts at Bottleneck
        # outputs.
        # ----------------------------------------------------

        for name, p in (
            self.free_model.named_parameters()
        ):

            if (
                name.startswith(
                    "conv1."
                )
                or
                name.startswith(
                    "bn1."
                )
            ):

                p.requires_grad = False


        # ----------------------------------------------------
        # TASK-2 CLASSIFIER
        # ----------------------------------------------------

        self.fc = nn.Linear(

            self.free_model.fc.in_features,

            NUM_CLASSES
        )


        self.fc.load_state_dict(
            copy.deepcopy(
                self.protected_model.fc.state_dict()
            )
        )


    # ========================================================
    # STEM
    # ========================================================

    def _stem(
        self,
        x
    ):

        x = self.protected_model.conv1(
            x
        )

        x = self.protected_model.bn1(
            x
        )

        x = self.protected_model.relu(
            x
        )

        x = self.protected_model.maxpool(
            x
        )

        return x


    # ========================================================
    # RUN ONE RESNET LAYER
    # ========================================================

    def _run_layer(
        self,
        protected_layer,
        free_layer,
        protected_x,
        free_x,
        layer_name
    ):

        protected_outputs = []

        combined_outputs = []


        for block_idx in range(
            len(protected_layer)
        ):


            protected_block = (
                protected_layer[
                    block_idx
                ]
            )


            free_block = (
                free_layer[
                    block_idx
                ]
            )


            key = (
                layer_name,
                block_idx
            )


            mask = (
                self.block_masks[key]
            )


            protected_mask = (
                mask
                .view(
                    1,
                    -1,
                    1,
                    1
                )
            )


            free_mask = (
                1.0
                -
                protected_mask
            )


            # ------------------------------------------------
            # FROZEN TASK-1 PROTECTED BRANCH
            # ------------------------------------------------

            with torch.no_grad():

                protected_out = (
                    protected_block(
                        protected_x
                    )
                )


                # Keep protected channels only.

                protected_out = (
                    protected_out
                    *
                    protected_mask
                )


            # ------------------------------------------------
            # TRAINABLE TASK-2 FREE BRANCH
            # ------------------------------------------------

            free_out = (
                free_block(
                    free_x
                )
            )


            # Keep free channels only.

            free_out = (
                free_out
                *
                free_mask
            )


            # ------------------------------------------------
            # CONTINUAL TRANSFER
            # ------------------------------------------------
            #
            # Task-2 receives:
            #
            #       Task-1 protected
            #              +
            #       Task-2 free
            # ------------------------------------------------

            combined = (
                protected_out
                +
                free_out
            )


            # Protected branch remains independent.

            protected_x = (
                protected_out
            )


            # Free branch can use the combined
            # protected + free representation.

            free_x = (
                combined
            )


            protected_outputs.append(
                protected_out
            )


            combined_outputs.append(
                combined
            )


        return (

            protected_x,

            free_x,

            protected_outputs,

            combined_outputs
        )


    # ========================================================
    # FORWARD
    # ========================================================

    def forward(
        self,
        x,
        return_intermediates=False
    ):

        # Frozen shared stem

        x = self._stem(x)


        protected_x = x

        free_x = x


        all_protected = []

        all_combined = []


        # ----------------------------------------------------
        # LAYER 1
        # ----------------------------------------------------

        (
            protected_x,
            free_x,
            protected_outputs,
            combined_outputs
        ) = self._run_layer(

            self.protected_model.layer1,

            self.free_model.layer1,

            protected_x,

            free_x,

            "layer1"
        )


        all_protected.extend(
            protected_outputs
        )

        all_combined.extend(
            combined_outputs
        )


        # ----------------------------------------------------
        # LAYER 2
        # ----------------------------------------------------

        (
            protected_x,
            free_x,
            protected_outputs,
            combined_outputs
        ) = self._run_layer(

            self.protected_model.layer2,

            self.free_model.layer2,

            protected_x,

            free_x,

            "layer2"
        )


        all_protected.extend(
            protected_outputs
        )

        all_combined.extend(
            combined_outputs
        )


        # ----------------------------------------------------
        # LAYER 3
        # ----------------------------------------------------

        (
            protected_x,
            free_x,
            protected_outputs,
            combined_outputs
        ) = self._run_layer(

            self.protected_model.layer3,

            self.free_model.layer3,

            protected_x,

            free_x,

            "layer3"
        )


        all_protected.extend(
            protected_outputs
        )

        all_combined.extend(
            combined_outputs
        )


        # ----------------------------------------------------
        # LAYER 4
        # ----------------------------------------------------

        (
            protected_x,
            free_x,
            protected_outputs,
            combined_outputs
        ) = self._run_layer(

            self.protected_model.layer4,

            self.free_model.layer4,

            protected_x,

            free_x,

            "layer4"
        )


        all_protected.extend(
            protected_outputs
        )

        all_combined.extend(
            combined_outputs
        )


        # ----------------------------------------------------
        # GLOBAL AVERAGE POOLING
        # ----------------------------------------------------

        pooled = (
            self.free_model.avgpool(
                free_x
            )
        )


        pooled = torch.flatten(
            pooled,
            1
        )


        # ----------------------------------------------------
        # TASK-2 CLASSIFIER
        # ----------------------------------------------------

        logits = self.fc(
            pooled
        )


        if return_intermediates:

            return {

                "logits":
                    logits,

                "protected_outputs":
                    all_protected,

                "combined_outputs":
                    all_combined,
            }


        return logits


# ============================================================
# 7.18 CREATE PROTECTED + FREE MODELS
# ============================================================

protected_model = (
    build_resnet50_binary()
)

free_model = (
    build_resnet50_binary()
)


protected_model.load_state_dict(
    state_dict,
    strict=True
)

free_model.load_state_dict(
    state_dict,
    strict=True
)


protected_model = (
    protected_model.to(DEVICE)
)

free_model = (
    free_model.to(DEVICE)
)


continual_model = (
    ContinualResNet50(

        protected_model=
            protected_model,

        free_model=
            free_model,

        block_masks=
            torch_block_masks
    )
    .to(DEVICE)
)


print(
    "\nContinual Task-2 model created."
)


# ============================================================
# 7.19 BATCHNORM SAFETY
# ============================================================

continual_model.eval()

protected_model.eval()

free_model.eval()


bn_modules = [

    m
    for m
    in continual_model.modules()

    if isinstance(
        m,
        nn.BatchNorm2d
    )
]


bn_not_eval = [

    m
    for m
    in bn_modules

    if m.training
]


print(
    "\nBatchNorm modules:",
    len(bn_modules)
)

print(
    "BatchNorm modules not in eval:",
    len(bn_not_eval)
)


batchnorm_eval_pass = (
    len(bn_not_eval) == 0
)


print(
    "BatchNorm check:",
    "PASS"
    if batchnorm_eval_pass
    else
    "FAIL"
)


assert batchnorm_eval_pass


# ============================================================
# 7.20 SNAPSHOT PROTECTED PARAMETERS
# ============================================================

protected_before = {

    name:
        p.detach().clone()

    for name, p
    in protected_model.named_parameters()
}


# ============================================================
# 7.21 SNAPSHOT FREE PARAMETERS
# ============================================================

free_before = {

    name:
        p.detach().clone()

    for name, p
    in free_model.named_parameters()

    if p.requires_grad
}


# ============================================================
# 7.22 TASK-2 BATCH
# ============================================================

task2_batch = next(
    iter(task2_train_loader)
)


images2, labels2 = (
    task2_batch
)


images2 = images2.to(
    DEVICE,
    non_blocking=True
)


labels2 = labels2.to(
    DEVICE,
    non_blocking=True
)


print(
    "\nTask-2 batch:"
)

print(
    images2.shape
)

print(
    labels2.shape
)


# ============================================================
# 7.23 TASK-1 REPRESENTATION BATCH
# ============================================================
#
# Use TRAIN data only.
#
# The Task-1 test set remains reserved for final evaluation.
# ============================================================

task1_batch = next(
    iter(task1_train_loader)
)


images1, labels1 = (
    task1_batch
)


images1 = images1.to(
    DEVICE,
    non_blocking=True
)


labels1 = labels1.to(
    DEVICE,
    non_blocking=True
)


print(
    "\nTask-1 representation batch:"
)

print(
    images1.shape
)

print(
    labels1.shape
)


# ============================================================
# 7.24 INITIAL TASK-2 FORWARD
# ============================================================

with torch.no_grad():

    initial_output = (
        continual_model(
            images2,
            return_intermediates=True
        )
    )


initial_logits = (
    initial_output["logits"]
)


initial_preds = (
    initial_logits.argmax(
        dim=1
    )
)


initial_class0 = int(
    (
        initial_preds == 0
    )
    .sum()
    .item()
)


initial_class1 = int(
    (
        initial_preds == 1
    )
    .sum()
    .item()
)


print(
    "\nInitial Task-2 prediction distribution:"
)

print(
    "Class 0:",
    initial_class0
)

print(
    "Class 1:",
    initial_class1
)


initial_not_collapsed = (
    initial_class0 > 0
    and
    initial_class1 > 0
)


print(
    "Initial prediction collapse:",
    "PASS"
    if initial_not_collapsed
    else
    "FAIL"
)


# ============================================================
# 7.25 PROTECTED REPRESENTATION BEFORE UPDATE
# ============================================================

with torch.no_grad():

    before_rep = (
        continual_model(
            images1,
            return_intermediates=True
        )
    )


before_protected = [

    tensor.detach().clone()

    for tensor
    in before_rep[
        "protected_outputs"
    ]
]


# ============================================================
# 7.26 VERIFY 16 BOTTLENECK OUTPUTS
# ============================================================

assert len(
    before_protected
) == 16


assert len(
    before_rep[
        "combined_outputs"
    ]
) == 16


print(
    "\nNumber of Bottleneck outputs:",
    len(before_protected)
)


print(
    "PASS — expected 16."
)


# ============================================================
# 7.27 FREE-BRANCH TRAINABILITY
# ============================================================

trainable_free_params = [

    p
    for p
    in free_model.parameters()

    if p.requires_grad
]


assert len(
    trainable_free_params
) > 0


print(
    "\nTrainable free-branch parameter tensors:",
    len(trainable_free_params)
)


# ============================================================
# 7.28 IMPORTANT:
#      SNAPSHOT FREE PARAMETERS AGAIN
# ============================================================

free_before = {

    name:
        p.detach().clone()

    for name, p
    in free_model.named_parameters()

    if p.requires_grad
}


# ============================================================
# 7.29 FREE-BRANCH-ONLY DIAGNOSTIC OPTIMIZER
# ============================================================
#
# This is NOT Task-2 training.
#
# It is only a mechanical test.
#
# We intentionally:
#
#   - use tiny SGD learning rate
#   - update FREE branch only
#   - do NOT update Task-2 classifier
#   - keep Task-1 branch frozen
#
# This isolates the free-unit pathway.
# ============================================================

DIAGNOSTIC_LR = 1e-7


print(
    "\nDiagnostic learning rate:",
    DIAGNOSTIC_LR
)


# ------------------------------------------------------------
# Freeze Task-2 classifier
# ------------------------------------------------------------

for p in (
    continual_model.fc.parameters()
):

    p.requires_grad = False


# ------------------------------------------------------------
# Create diagnostic optimizer
# ------------------------------------------------------------

diagnostic_optimizer = (
    torch.optim.SGD(

        trainable_free_params,

        lr=DIAGNOSTIC_LR
    )
)


# ============================================================
# 7.30 FORWARD
# ============================================================

diagnostic_optimizer.zero_grad(
    set_to_none=True
)


output = continual_model(
    images2,
    return_intermediates=True
)


logits = output[
    "logits"
]


# ============================================================
# 7.31 LOSS
# ============================================================

loss = nn.CrossEntropyLoss()(
    logits,
    labels2
)


print(
    "\nMechanical CE loss:",
    float(loss.item())
)


assert torch.isfinite(
    loss
), (
    "Mechanical loss is NaN or Inf."
)


# ============================================================
# 7.32 BACKWARD
# ============================================================

loss.backward()


# ============================================================
# 7.33 PROTECTED GRADIENT CHECK
# ============================================================

protected_grad_squared = 0.0

protected_nonzero = 0


for p in (
    protected_model.parameters()
):

    if p.grad is not None:

        g = (
            p.grad
            .detach()
            .float()
        )


        protected_grad_squared += (
            g.norm(2).item() ** 2
        )


        protected_nonzero += int(
            (
                g != 0
            )
            .sum()
            .item()
        )


protected_grad_norm = (
    protected_grad_squared ** 0.5
)


# ============================================================
# 7.34 FREE GRADIENT CHECK
# ============================================================

free_grad_squared = 0.0

free_nonzero = 0


for p in (
    free_model.parameters()
):

    if p.grad is not None:

        g = (
            p.grad
            .detach()
            .float()
        )


        free_grad_squared += (
            g.norm(2).item() ** 2
        )


        free_nonzero += int(
            (
                g != 0
            )
            .sum()
            .item()
        )


free_grad_norm = (
    free_grad_squared ** 0.5
)


print(
    "\nGradient checks:"
)


print(
    "Protected gradient norm:",
    protected_grad_norm
)


print(
    "Free gradient norm:",
    free_grad_norm
)


print(
    "Protected nonzero gradient elements:",
    protected_nonzero
)


print(
    "Free nonzero gradient elements:",
    free_nonzero
)


# ============================================================
# 7.35 ONE TINY FREE-BRANCH UPDATE
# ============================================================

diagnostic_optimizer.step()


# ============================================================
# 7.36 PROTECTED PARAMETER CHANGE
# ============================================================

protected_max_change = 0.0


for name, p in (
    protected_model.named_parameters()
):

    change = (

        p.detach()
        -
        protected_before[name]

    ).abs().max().item()


    protected_max_change = max(

        protected_max_change,

        change
    )


# ============================================================
# 7.37 FREE PARAMETER CHANGE
# ============================================================

free_max_change = 0.0


for name, p in (
    free_model.named_parameters()
):

    if name in free_before:

        change = (

            p.detach()
            -
            free_before[name]

        ).abs().max().item()


        free_max_change = max(

            free_max_change,

            change
        )


print(
    "\nParameter changes after diagnostic update:"
)


print(
    "Protected max change:",
    protected_max_change
)


print(
    "Free branch max change:",
    free_max_change
)


# ============================================================
# 7.38 PROTECTED REPRESENTATION AFTER UPDATE
# ============================================================

with torch.no_grad():

    after_rep = (
        continual_model(
            images1,
            return_intermediates=True
        )
    )


after_protected = [

    tensor.detach().clone()

    for tensor
    in after_rep[
        "protected_outputs"
    ]
]


# ============================================================
# 7.39 PROTECTED REPRESENTATION STABILITY
# ============================================================

representation_differences = []


for before, after in zip(

    before_protected,

    after_protected

):

    diff = (

        before
        -
        after

    ).abs().max().item()


    representation_differences.append(
        diff
    )


max_protected_rep_difference = max(
    representation_differences
)


print(
    "\nProtected representation differences:"
)


for idx, diff in enumerate(
    representation_differences
):

    print(
        f"  Bottleneck {idx:02d}: "
        f"{diff:.12e}"
    )


print(
    "\nMaximum protected representation difference:",
    max_protected_rep_difference
)


# ============================================================
# 7.40 TENSOR SHAPE CHECK
# ============================================================

shape_valid = all(

    tensor.ndim == 4

    for tensor
    in after_rep[
        "combined_outputs"
    ]
)


print(
    "\nTensor shape check:",
    "PASS"
    if shape_valid
    else
    "FAIL"
)


# ============================================================
# 7.41 FINITE VALUE CHECK
# ============================================================

finite_valid = all(

    torch.isfinite(
        tensor
    ).all().item()

    for tensor
    in after_rep[
        "combined_outputs"
    ]
)


print(
    "Finite-value check:",
    "PASS"
    if finite_valid
    else
    "FAIL"
)


# ============================================================
# 7.42 RESIDUAL STRUCTURE CHECK
# ============================================================

residual_block_count = len(
    after_rep[
        "combined_outputs"
    ]
)


residual_structure_valid = (
    residual_block_count == 16
)


print(
    "Residual blocks executed:",
    residual_block_count
)


print(
    "Residual structure:",
    "PASS"
    if residual_structure_valid
    else
    "FAIL"
)


# ============================================================
# 7.43 POST-UPDATE TASK-2 PREDICTIONS
# ============================================================

with torch.no_grad():

    final_output = (
        continual_model(
            images2,
            return_intermediates=True
        )
    )


final_logits = (
    final_output["logits"]
)


final_preds = (
    final_logits.argmax(
        dim=1
    )
)


final_class0 = int(
    (
        final_preds == 0
    )
    .sum()
    .item()
)


final_class1 = int(
    (
        final_preds == 1
    )
    .sum()
    .item()
)


print(
    "\nPost-update Task-2 predictions:"
)


print(
    "Class 0:",
    final_class0
)


print(
    "Class 1:",
    final_class1
)


post_update_not_collapsed = (

    final_class0 > 0

    and

    final_class1 > 0
)


print(
    "Post-update prediction collapse:",
    "PASS"
    if post_update_not_collapsed
    else
    "FAIL"
)


# ============================================================
# 7.44 FINAL SANITY CHECKS
# ============================================================

checks = {

    "task1_parameters_unchanged":
        protected_max_change < 1e-12,

    "task2_free_parameters_changed":
        free_max_change > 0,

    "task1_representation_stable":
        max_protected_rep_difference < 1e-12,

    "protected_gradient_zero":
        protected_grad_norm < 1e-12,

    "free_gradient_nonzero":
        free_grad_norm > 0,

    "batchnorm_eval":
        batchnorm_eval_pass,

    "tensor_shapes_valid":
        shape_valid,

    "finite_values":
        finite_valid,

    "residual_structure_valid":
        residual_structure_valid,

    "initial_predictions_not_collapsed":
        initial_not_collapsed,

    "post_update_predictions_not_collapsed":
        post_update_not_collapsed,
}


# ============================================================
# 7.45 PRINT FINAL RESULT
# ============================================================

print("\n")

print("=" * 75)

print(
    "STEP 7 — FINAL SANITY CHECK RESULTS"
)

print("=" * 75)


for name, passed in checks.items():

    print(
        f"{name:48s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


all_pass = all(
    checks.values()
)


print(
    "\n" + "=" * 75
)


if all_pass:

    print(
        "STEP 7 STATUS: PASS"
    )

    print()

    print(
        "Task-1 protected branch : FROZEN + ACTIVE"
    )

    print(
        "Task-2 free branch      : TRAINABLE + ACTIVE"
    )

    print(
        "Protected representation: STABLE"
    )

    print(
        "Free representation     : TRAINABLE"
    )

    print()

    print(
        "=================================================="
    )

    print(
        "SAFE TO PROCEED TO STEP 8"
    )

    print(
        "=================================================="

    )

else:

    print(
        "STEP 7 STATUS: FAIL"
    )

    print()

    print(
        "DO NOT START TASK-2 GCE."
    )


# ============================================================
# 7.46 SAVE STEP-7 RESULTS
# ============================================================

step7_results = {

    "step": 7,

    "version": "v5",

    "status":
        "PASS"
        if all_pass
        else
        "FAIL",

    "implementation":
        "fixed_shape_dual_branch_continual_transfer",

    "diagnostic_update":
        "tiny_SGD_free_branch_only",

    "diagnostic_learning_rate":
        DIAGNOSTIC_LR,

    "protected_units":
        int(total_protected),

    "free_units":
        int(total_free),

    "total_units":
        int(
            total_protected
            +
            total_free
        ),

    "mechanical_loss":
        float(
            loss.item()
        ),

    "protected_gradient_norm":
        float(
            protected_grad_norm
        ),

    "free_gradient_norm":
        float(
            free_grad_norm
        ),

    "protected_nonzero_gradients":
        int(
            protected_nonzero
        ),

    "free_nonzero_gradients":
        int(
            free_nonzero
        ),

    "protected_max_parameter_change":
        float(
            protected_max_change
        ),

    "free_max_parameter_change":
        float(
            free_max_change
        ),

    "max_protected_representation_difference":
        float(
            max_protected_rep_difference
        ),

    "initial_prediction_class0":
        int(
            initial_class0
        ),

    "initial_prediction_class1":
        int(
            initial_class1
        ),

    "post_update_prediction_class0":
        int(
            final_class0
        ),

    "post_update_prediction_class1":
        int(
            final_class1
        ),

    "checks": {

        k: bool(v)

        for k, v
        in checks.items()
    },
}


with open(
    STEP7_RESULT_PATH,
    "w"
) as f:

    json.dump(
        step7_results,
        f,
        indent=2
    )


print(
    "\nStep-7 result saved to:"
)

print(
    STEP7_RESULT_PATH
)


print(
    "\nSTEP 7 COMPLETE."
)


if all_pass:

    print(
        "NEXT: STEP 8 — TASK-2 GCE"
    )

else:

    print(
        "DO NOT PROCEED TO STEP 8."
    )


print(
    "=" * 75
)

STEP 7 — TASK-2 MECHANICAL SANITY TEST v5
Device: cuda
Task-1 checkpoint: /kaggle/working/BiasPruner_HAM10000/task1_final.pt
Pruning plan:      /kaggle/working/BiasPruner_HAM10000/task1_pruning_plan.csv

Required files: PASS

Task-1 checkpoint keys:
dict_keys(['model_state_dict', 'alpha_raw', 'alpha', 'pruning_masks', 'gamma', 'tau', 'q', 'num_total_units', 'num_pruned', 'num_retained', 'best_epoch', 'best_val_accuracy', 'best_val_abs_EOD', 'seed', 'task', 'implementation_note'])

Task-1 model loaded successfully.
Task-1 parameters: 23512130

Pruning-plan columns:
['global_unit_id', 'layer', 'block_index', 'channel', 'bias_score', 'bias_rank', 'prune', 'status']
Pruning-plan shape: (15104, 8)

Expected total units: 15104

Pruning-plan format: PASS

Unique values in 'prune': [0, 1]

Ownership conversion completed.
Internal meaning:
    1 = protected / retained
    0 = free / pruned

Per-block ownership:
  layer1 block 0: protected=  21, free= 235, total= 256
  layer1 block 1: protected=

In [19]:
# ============================================================
# STEP 8 — TASK-2 GCE TRAINING
# ============================================================
#
# Task 2:
#     BCC   -> 0
#     AKIEC -> 1
#
# Continual-learning mechanism validated in STEP 7:
#
#     Task-1 protected branch -> frozen + active
#     Task-2 free branch      -> trainable + active
#
# GCE:
#     q = 0.7
#
# Training:
#     max epochs = 200
#     early stopping
#     batch size = 32
#     learning rate = 1e-4
#
# IMPORTANT:
#     Task-2 TEST SET IS NOT USED HERE.
#
# Validation is used only for checkpoint selection /
# early stopping.
# ============================================================


# ============================================================
# 8.1 CONFIGURATION
# ============================================================

import os
import json
import time
import copy
import numpy as np
import pandas as pd

import torch
import torch.nn as nn


print("=" * 75)
print("STEP 8 — TASK-2 GCE TRAINING")
print("=" * 75)


DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

PROJECT_DIR = "/kaggle/working/BiasPruner_HAM10000"

STEP7_RESULT_PATH = os.path.join(
    PROJECT_DIR,
    "task2_mechanical_sanity_v5.json"
)

TASK2_GCE_CKPT = os.path.join(
    PROJECT_DIR,
    "task2_gce_best.pt"
)

TASK2_GCE_HISTORY = os.path.join(
    PROJECT_DIR,
    "task2_gce_history.csv"
)

TASK2_GCE_CONFIG = os.path.join(
    PROJECT_DIR,
    "task2_gce_config.json"
)


# ============================================================
# HYPERPARAMETERS
# ============================================================

Q = 0.7

MAX_EPOCHS = 200

PATIENCE = 15

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 0.0

NUM_CLASSES = 2


print(
    "Device:",
    DEVICE
)

print(
    "GCE q:",
    Q
)

print(
    "Maximum epochs:",
    MAX_EPOCHS
)

print(
    "Early stopping patience:",
    PATIENCE
)

print(
    "Learning rate:",
    LEARNING_RATE
)

print(
    "Weight decay:",
    WEIGHT_DECAY
)


# ============================================================
# 8.2 VERIFY STEP 7 PASSED
# ============================================================

assert os.path.exists(
    STEP7_RESULT_PATH
), (
    "Step-7 result file not found."
)

with open(
    STEP7_RESULT_PATH,
    "r"
) as f:

    step7_results = json.load(f)


print(
    "\nStep-7 recorded status:",
    step7_results["status"]
)


assert (
    step7_results["status"]
    == "PASS"
), (
    "Step 7 did not PASS. "
    "Do not start Task-2 GCE."
)


print(
    "Step-7 verification: PASS"
)


# ============================================================
# 8.3 VERIFY CONTINUAL MODEL EXISTS
# ============================================================
#
# ContinualResNet50 was defined in Step 7.
# We intentionally reuse that exact implementation.
# ============================================================

assert (
    "ContinualResNet50"
    in globals()
), (
    "ContinualResNet50 is not defined. "
    "Run the successful Step-7 cell first."
)


assert (
    "block_masks"
    in globals()
), (
    "block_masks is not available. "
    "Run Step 7 first."
)


assert (
    "torch_block_masks"
    in globals()
), (
    "torch_block_masks is not available. "
    "Run Step 7 first."
)


# ============================================================
# 8.4 REBUILD TASK-1 REFERENCE + TASK-2 FREE MODELS
# ============================================================
#
# We load the SAME Task-1 final checkpoint used in Step 7.
#
# This ensures Step 8 starts from exactly the validated
# continual-learning initialization.
# ============================================================

TASK1_FINAL_CKPT = os.path.join(
    PROJECT_DIR,
    "task1_final.pt"
)

assert os.path.exists(
    TASK1_FINAL_CKPT
), (
    "Task-1 final checkpoint missing."
)


checkpoint = torch.load(
    TASK1_FINAL_CKPT,
    map_location="cpu"
)


if "model_state_dict" in checkpoint:

    state_dict = checkpoint[
        "model_state_dict"
    ]

elif "state_dict" in checkpoint:

    state_dict = checkpoint[
        "state_dict"
    ]

else:

    state_dict = checkpoint


# ------------------------------------------------------------
# Protected Task-1 branch
# ------------------------------------------------------------

protected_model = (
    build_resnet50_binary()
)

protected_model.load_state_dict(
    state_dict,
    strict=True
)

protected_model = (
    protected_model.to(DEVICE)
)


# ------------------------------------------------------------
# Trainable Task-2 free branch
# ------------------------------------------------------------

free_model = (
    build_resnet50_binary()
)

free_model.load_state_dict(
    state_dict,
    strict=True
)

free_model = (
    free_model.to(DEVICE)
)


# ============================================================
# 8.5 BUILD CONTINUAL TASK-2 MODEL
# ============================================================

model = ContinualResNet50(

    protected_model=
        protected_model,

    free_model=
        free_model,

    block_masks=
        torch_block_masks
)

model = model.to(DEVICE)


print(
    "\nContinual Task-2 model rebuilt."
)


# ============================================================
# 8.6 VERIFY PARAMETER OWNERSHIP
# ============================================================

protected_trainable = sum(

    p.numel()

    for p
    in model.protected_model.parameters()

    if p.requires_grad
)


free_trainable = sum(

    p.numel()

    for p
    in model.free_model.parameters()

    if p.requires_grad
)


classifier_trainable = sum(

    p.numel()

    for p
    in model.fc.parameters()

    if p.requires_grad
)


print(
    "\nTrainable parameter ownership:"
)

print(
    "Protected Task-1 parameters:",
    protected_trainable
)

print(
    "Free Task-2 parameters:",
    free_trainable
)

print(
    "Task-2 classifier parameters:",
    classifier_trainable
)


# Protected branch MUST be completely frozen.

assert (
    protected_trainable == 0
), (
    "Protected Task-1 branch contains trainable parameters."
)


assert (
    free_trainable > 0
), (
    "No trainable Task-2 free parameters found."
)


assert (
    classifier_trainable > 0
), (
    "Task-2 classifier is not trainable."
)


print(
    "Parameter ownership: PASS"
)


# ============================================================
# 8.7 DATASET SANITY CHECK
# ============================================================
#
# Expected:
#
# Task 2 = BCC + AKIEC
#
# train = 587
# val   = 165
# test  = 89
#
# Test is NOT used below.
# ============================================================

print(
    "\nTask-2 dataset sizes:"
)

print(
    "Train:",
    len(task2_train_loader.dataset)
)

print(
    "Validation:",
    len(task2_val_loader.dataset)
)

print(
    "Test:",
    len(task2_test_loader.dataset)
)


assert (
    len(task2_train_loader.dataset)
    == 587
), (
    "Unexpected Task-2 train size."
)


assert (
    len(task2_val_loader.dataset)
    == 165
), (
    "Unexpected Task-2 validation size."
)


assert (
    len(task2_test_loader.dataset)
    == 89
), (
    "Unexpected Task-2 test size."
)


print(
    "Task-2 split sizes: PASS"
)


# ============================================================
# 8.8 GCE FUNCTION
# ============================================================
#
# Generalized Cross Entropy:
#
#       L_GCE =
#           (1 - p_y^q) / q
#
# where:
#
#       p_y = predicted probability of true class
#
# and:
#
#       q = 0.7
#
# For q -> 0, this approaches CE.
# ============================================================

def generalized_cross_entropy(
    logits,
    targets,
    q=0.7
):

    probabilities = torch.softmax(
        logits,
        dim=1
    )

    true_probabilities = (
        probabilities[
            torch.arange(
                targets.size(0),
                device=targets.device
            ),
            targets
        ]
    )


    # Numerical stability

    true_probabilities = torch.clamp(
        true_probabilities,
        min=1e-8,
        max=1.0
    )


    gce = (
        1.0
        -
        true_probabilities.pow(q)
    ) / q


    return gce.mean()


# ============================================================
# 8.9 VALIDATION METRICS
# ============================================================
#
# Validation is used for:
#
#   - checkpoint selection
#   - early stopping
#
# Test is NOT touched.
# ============================================================

def evaluate_gce_validation(
    model,
    loader,
    device
):

    model.eval()

    total_loss = 0.0

    total_samples = 0

    correct = 0

    all_predictions = []

    all_targets = []


    with torch.no_grad():

        for images, targets in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            targets = targets.to(
                device,
                non_blocking=True
            )


            logits = model(
                images
            )


            loss = generalized_cross_entropy(
                logits,
                targets,
                q=Q
            )


            batch_size = (
                targets.size(0)
            )


            total_loss += (
                loss.item()
                *
                batch_size
            )


            total_samples += (
                batch_size
            )


            predictions = (
                logits.argmax(
                    dim=1
                )
            )


            correct += int(
                (
                    predictions
                    ==
                    targets
                )
                .sum()
                .item()
            )


            all_predictions.extend(
                predictions
                .cpu()
                .numpy()
                .tolist()
            )


            all_targets.extend(
                targets
                .cpu()
                .numpy()
                .tolist()
            )


    avg_loss = (
        total_loss /
        total_samples
    )


    accuracy = (
        correct /
        total_samples
    )


    all_predictions = np.asarray(
        all_predictions
    )

    all_targets = np.asarray(
        all_targets
    )


    # --------------------------------------------------------
    # Binary class-specific metrics
    # --------------------------------------------------------

    metrics = {

        "loss": float(
            avg_loss
        ),

        "accuracy": float(
            accuracy
        ),
    }


    for cls in [0, 1]:

        cls_mask = (
            all_targets == cls
        )

        cls_count = int(
            cls_mask.sum()
        )


        if cls_count > 0:

            cls_accuracy = (
                (
                    all_predictions[
                        cls_mask
                    ]
                    ==
                    cls
                )
                .mean()
            )

            metrics[
                f"class_{cls}_accuracy"
            ] = float(
                cls_accuracy
            )

        else:

            metrics[
                f"class_{cls}_accuracy"
            ] = float("nan")


    # --------------------------------------------------------
    # TPR / EOD
    # --------------------------------------------------------

    tpr_values = {}

    for cls in [0, 1]:

        cls_mask = (
            all_targets == cls
        )

        cls_count = int(
            cls_mask.sum()
        )


        if cls_count > 0:

            tpr = (
                (
                    all_predictions[
                        cls_mask
                    ]
                    ==
                    cls
                )
                .mean()
            )

            tpr_values[
                cls
            ] = float(tpr)

        else:

            tpr_values[
                cls
            ] = float("nan")


    # For Task-2:
    #
    # class 0 = BCC
    # class 1 = AKIEC
    #
    # EOD = TPR(class 0) - TPR(class 1)

    eod = (
        tpr_values[0]
        -
        tpr_values[1]
    )


    metrics[
        "tpr_class_0"
    ] = tpr_values[0]

    metrics[
        "tpr_class_1"
    ] = tpr_values[1]

    metrics[
        "EOD"
    ] = float(eod)

    metrics[
        "abs_EOD"
    ] = float(abs(eod))


    return metrics


# ============================================================
# 8.10 OPTIMIZER
# ============================================================
#
# IMPORTANT:
#
# The protected Task-1 branch is NOT included.
#
# Train:
#
#   - Task-2 free branch
#   - Task-2 classifier
#
# Learning rate = 1e-4
#
# This is actual training, unlike Step 7's 1e-7
# diagnostic update.
# ============================================================

trainable_parameters = [

    p

    for p
    in model.parameters()

    if p.requires_grad
]


print(
    "\nTotal trainable parameters:",
    sum(
        p.numel()
        for p in trainable_parameters
    )
)


optimizer = torch.optim.Adam(

    trainable_parameters,

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY
)


# ============================================================
# 8.11 TRAINING LOOP
# ============================================================

history = []

best_val_accuracy = -float("inf")

best_val_abs_EOD = float("inf")

best_epoch = -1

epochs_without_improvement = 0

training_start = time.time()


for epoch in range(
    1,
    MAX_EPOCHS + 1
):


    epoch_start = time.time()


    # --------------------------------------------------------
    # TRAIN MODE
    # --------------------------------------------------------

    model.train()


    # --------------------------------------------------------
    # CRITICAL:
    #
    # The protected Task-1 branch must remain frozen and
    # should remain in eval mode.
    #
    # The free branch can train.
    # --------------------------------------------------------

    model.protected_model.eval()

    model.free_model.train()


    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    running_loss = 0.0

    running_correct = 0

    running_samples = 0


    for images, targets in (
        task2_train_loader
    ):

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        targets = targets.to(
            DEVICE,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        logits = model(
            images
        )


        loss = generalized_cross_entropy(
            logits,
            targets,
            q=Q
        )


        if not torch.isfinite(
            loss
        ):

            raise RuntimeError(
                f"Non-finite loss at epoch "
                f"{epoch}: {loss.item()}"
            )


        loss.backward()


        # ----------------------------------------------------
        # Confirm protected branch still receives no gradient
        # ----------------------------------------------------

        protected_has_gradient = any(

            p.grad is not None

            for p
            in model.protected_model.parameters()
        )


        if protected_has_gradient:

            raise RuntimeError(
                "ERROR: protected Task-1 branch "
                "received a gradient."
            )


        optimizer.step()


        batch_size = (
            targets.size(0)
        )


        running_loss += (
            loss.item()
            *
            batch_size
        )


        predictions = (
            logits.argmax(
                dim=1
            )
        )


        running_correct += int(
            (
                predictions
                ==
                targets
            )
            .sum()
            .item()
        )


        running_samples += (
            batch_size
        )


    train_loss = (
        running_loss /
        running_samples
    )


    train_accuracy = (
        running_correct /
        running_samples
    )


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    val_metrics = (
        evaluate_gce_validation(
            model,
            task2_val_loader,
            DEVICE
        )
    )


    val_accuracy = (
        val_metrics[
            "accuracy"
        ]
    )


    val_abs_EOD = (
        val_metrics[
            "abs_EOD"
        ]
    )


    epoch_time = (
        time.time()
        -
        epoch_start
    )


    # --------------------------------------------------------
    # CHECKPOINT SELECTION
    # --------------------------------------------------------
    #
    # Primary:
    #       highest validation accuracy
    #
    # Tie-break:
    #       lower absolute EOD
    #
    # This is an explicit implementation decision because
    # the paper states validation ACC/EOD but does not give
    # a fully specified algorithm for combining them.
    # --------------------------------------------------------

    improved = False


    if val_accuracy > (
        best_val_accuracy
        + 1e-12
    ):

        improved = True


    elif abs(
        val_accuracy
        -
        best_val_accuracy
    ) <= 1e-12:

        if val_abs_EOD < (
            best_val_abs_EOD
            - 1e-12
        ):

            improved = True


    # --------------------------------------------------------
    # Save best model
    # --------------------------------------------------------

    if improved:

        best_val_accuracy = (
            val_accuracy
        )

        best_val_abs_EOD = (
            val_abs_EOD
        )

        best_epoch = (
            epoch
        )

        epochs_without_improvement = 0


        # --------------------------------------------
        # Save COMPLETE continual-learning state
        # --------------------------------------------

        checkpoint_to_save = {

            "protected_model_state_dict":
                copy.deepcopy(
                    model.protected_model.state_dict()
                ),

            "free_model_state_dict":
                copy.deepcopy(
                    model.free_model.state_dict()
                ),

            "fc_state_dict":
                copy.deepcopy(
                    model.fc.state_dict()
                ),

            "q":
                Q,

            "learning_rate":
                LEARNING_RATE,

            "weight_decay":
                WEIGHT_DECAY,

            "epoch":
                epoch,

            "best_val_accuracy":
                best_val_accuracy,

            "best_val_abs_EOD":
                best_val_abs_EOD,

            "task":
                "Task2_BCC_AKIEC",

            "task_mapping": {

                "BCC": 0,

                "AKIEC": 1,
            },

            "protected_units":
                6041,

            "free_units":
                9063,

            "implementation":
                "fixed_shape_dual_branch_continual_transfer",

            "implementation_note": (
                "Frozen Task-1 protected branch "
                "combined with trainable Task-2 "
                "free branch at Bottleneck outputs."
            ),

            "seed":
                42,
        }


        torch.save(
            checkpoint_to_save,
            TASK2_GCE_CKPT
        )


    else:

        epochs_without_improvement += 1


    # --------------------------------------------------------
    # Save history row
    # --------------------------------------------------------

    history_row = {

        "epoch":
            epoch,

        "train_loss":
            train_loss,

        "train_accuracy":
            train_accuracy,

        "val_loss":
            val_metrics[
                "loss"
            ],

        "val_accuracy":
            val_accuracy,

        "val_class_0_accuracy":
            val_metrics[
                "class_0_accuracy"
            ],

        "val_class_1_accuracy":
            val_metrics[
                "class_1_accuracy"
            ],

        "val_tpr_class_0":
            val_metrics[
                "tpr_class_0"
            ],

        "val_tpr_class_1":
            val_metrics[
                "tpr_class_1"
            ],

        "val_EOD":
            val_metrics[
                "EOD"
            ],

        "val_abs_EOD":
            val_abs_EOD,

        "epoch_time_seconds":
            epoch_time,

        "is_best":
            improved,
    }


    history.append(
        history_row
    )


    # --------------------------------------------------------
    # PRINT
    # --------------------------------------------------------

    print(
        f"Epoch {epoch:03d} | "
        f"Train Loss {train_loss:.4f} | "
        f"Train Acc {train_accuracy:.4f} | "
        f"Val Loss {val_metrics['loss']:.4f} | "
        f"Val Acc {val_accuracy:.4f} | "
        f"Val |EOD| {val_abs_EOD:.4f} | "
        f"{epoch_time:.1f}s"
        +
        (
            "  <-- BEST"
            if improved
            else ""
        )
    )


    # --------------------------------------------------------
    # EARLY STOPPING
    # --------------------------------------------------------

    if (
        epochs_without_improvement
        >= PATIENCE
    ):

        print()
        print(
            "Early stopping triggered."
        )

        print(
            "Epoch:",
            epoch
        )

        print(
            "Best epoch:",
            best_epoch
        )

        break


# ============================================================
# 8.12 TRAINING FINISHED
# ============================================================

total_training_time = (
    time.time()
    -
    training_start
)


print()
print("=" * 75)
print("TASK-2 GCE TRAINING FINISHED")
print("=" * 75)


print(
    "Best epoch:",
    best_epoch
)

print(
    "Best validation accuracy:",
    best_val_accuracy
)

print(
    "Best validation |EOD|:",
    best_val_abs_EOD
)

print(
    "Total training time:",
    total_training_time,
    "seconds"
)


# ============================================================
# 8.13 SAVE HISTORY
# ============================================================

history_df = pd.DataFrame(
    history
)

history_df.to_csv(
    TASK2_GCE_HISTORY,
    index=False
)


print(
    "\nHistory saved:"
)

print(
    TASK2_GCE_HISTORY
)


# ============================================================
# 8.14 SAVE CONFIGURATION
# ============================================================

config = {

    "step":
        8,

    "task":
        "Task2_BCC_AKIEC",

    "task_mapping": {

        "BCC": 0,

        "AKIEC": 1,
    },

    "q":
        Q,

    "max_epochs":
        MAX_EPOCHS,

    "early_stopping_patience":
        PATIENCE,

    "learning_rate":
        LEARNING_RATE,

    "weight_decay":
        WEIGHT_DECAY,

    "batch_size":
        32,

    "best_epoch":
        best_epoch,

    "best_val_accuracy":
        best_val_accuracy,

    "best_val_abs_EOD":
        best_val_abs_EOD,

    "epochs_completed":
        len(history),

    "training_time_seconds":
        total_training_time,

    "protected_units":
        6041,

    "free_units":
        9063,

    "test_used":
        False,

    "implementation":
        "fixed_shape_dual_branch_continual_transfer",

    "implementation_note": (
        "Task-1 protected branch remains frozen "
        "and active. Task-2 free branch is "
        "trainable. Task-2 representation is "
        "the composition of protected Task-1 "
        "units and free Task-2 units."
    ),
}


with open(
    TASK2_GCE_CONFIG,
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=2
    )


print(
    "Configuration saved:"
)

print(
    TASK2_GCE_CONFIG
)


# ============================================================
# 8.15 LOAD BEST CHECKPOINT
# ============================================================
#
# We restore the BEST validation checkpoint before finishing
# Step 8.
#
# Still NO test evaluation.
# ============================================================

assert os.path.exists(
    TASK2_GCE_CKPT
), (
    "Best Task-2 GCE checkpoint was not created."
)


best_checkpoint = torch.load(
    TASK2_GCE_CKPT,
    map_location=DEVICE
)


model.protected_model.load_state_dict(
    best_checkpoint[
        "protected_model_state_dict"
    ],
    strict=True
)


model.free_model.load_state_dict(
    best_checkpoint[
        "free_model_state_dict"
    ],
    strict=True
)


model.fc.load_state_dict(
    best_checkpoint[
        "fc_state_dict"
    ],
    strict=True
)


# Re-establish freezing.

for p in (
    model.protected_model.parameters()
):

    p.requires_grad = False


# Re-establish frozen stem.

for name, p in (
    model.free_model.named_parameters()
):

    if (
        name.startswith("conv1.")
        or
        name.startswith("bn1.")
    ):

        p.requires_grad = False


print(
    "\nBest Task-2 GCE checkpoint restored."
)


# ============================================================
# 8.16 FINAL PROTECTED-BRANCH INTEGRITY CHECK
# ============================================================

protected_trainable_after = sum(

    p.numel()

    for p
    in model.protected_model.parameters()

    if p.requires_grad
)


assert (
    protected_trainable_after == 0
)


print(
    "Protected Task-1 branch remains frozen: PASS"
)


# ============================================================
# 8.17 IMPORTANT: NO TEST EVALUATION
# ============================================================

print()
print("=" * 75)

print(
    "STEP 8 STATUS: COMPLETE"
)

print("=" * 75)

print(
    "Best Task-2 GCE checkpoint:"
)

print(
    TASK2_GCE_CKPT
)

print()
print(
    "Task-2 TEST SET WAS NOT EVALUATED."
)

print(
    "Test evaluation will happen only at the "
    "appropriate final evaluation stage."
)

print()
print(
    "Next: STEP 9 — TASK-2 BIAS PRUNING + WCE"
)

print("=" * 75)

STEP 8 — TASK-2 GCE TRAINING
Device: cuda
GCE q: 0.7
Maximum epochs: 200
Early stopping patience: 15
Learning rate: 0.0001
Weight decay: 0.0

Step-7 recorded status: PASS
Step-7 verification: PASS

Continual Task-2 model rebuilt.

Trainable parameter ownership:
Protected Task-1 parameters: 0
Free Task-2 parameters: 23502594
Task-2 classifier parameters: 4098
Parameter ownership: PASS

Task-2 dataset sizes:
Train: 587
Validation: 165
Test: 89
Task-2 split sizes: PASS

Total trainable parameters: 23506692


OutOfMemoryError: CUDA out of memory. Tried to allocate 50.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 50.81 MiB is free. Including non-PyTorch memory, this process has 14.51 GiB memory in use. Of the allocated memory 13.56 GiB is allocated by PyTorch, and 829.05 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)